<a href="https://colab.research.google.com/github/diegozuu/asistente-para-planificacion-de-toma-de-ramos/blob/main/PROTOTIPO%205%20EVALUACION%203/Asistente_Horarios_Logs_y_DASHBOARD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. INSTALACIÓN E IMPORTACIÓN DE LIBRERÍAS

Instala las dependencias necesarias para OCR (`tesseract-ocr`), interfaz web (`gradio`), visualización (`plotly`), procesamiento de datos (`pandas`, `numpy`, `opencv`) y conexión con el modelo LLM (`openai`).

In [1]:
!apt-get install -y tesseract-ocr tesseract-ocr-spa -q
!pip install gradio openai pandas pytesseract Pillow opencv-python-headless plotly -q

import os
import re
import json
import time
import difflib
import unicodedata
from collections import Counter
import numpy as np
import cv2
import pandas as pd
import gradio as gr
import plotly.graph_objects as go
from openai import OpenAI
from google.colab import userdata
import pytesseract
from PIL import Image

print("[Exito] Librerias importadas correctamente con soporte para imagenes (OCR + vision) y Plotly")

Reading package lists...
Building dependency tree...
Reading state information...
tesseract-ocr is already the newest version (5.3.4-1build5).
The following NEW packages will be installed:
  tesseract-ocr-spa
0 upgraded, 1 newly installed, 0 to remove and 0 not upgraded.
Need to get 1,065 kB of archives.
After this operation, 2,309 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble/universe amd64 tesseract-ocr-spa all 1:4.1.0-2 [1,065 kB]
Fetched 1,065 kB in 19s (55.9 kB/s)
Selecting previously unselected package tesseract-ocr-spa.
(Reading database ... 122813 files and directories currently installed.)
Preparing to unpack .../tesseract-ocr-spa_1%3a4.1.0-2_all.deb ...
Unpacking tesseract-ocr-spa (1:4.1.0-2) ...
Setting up tesseract-ocr-spa (1:4.1.0-2) ...
[Exito] Librerias importadas correctamente con soporte para imagenes (OCR + vision) y Plotly


# 2. CONEXIÓN A LA API Y CARGA DEL EXCEL

**¿Qué hace esta celda?**
Establece dos conexiones: con el servicio de inteligencia artificial (**Groq** con `openai/gpt-oss-120b`) y con el único archivo base que se sube al Colab: `HORARIOS_SIMULADOS_CORREGIDO.xlsx` (`BASE` y `OPTATIVOS COMPETENCIAS GLOBALES`). Las mallas curriculares ya no se cargan aquí, sino dinámicamente desde la interfaz web.

In [2]:
try:
    cliente_ai = OpenAI(
        api_key=userdata.get('OPENAI_API_KEY'),
        base_url="https://api.groq.com/openai/v1"
    )
    modelo_elegido = "openai/gpt-oss-120b"
    print("[Exito] API key y modelo IA funcionando correctamente")
except Exception as e:
    print(f"[Error] Al conectar con la API: {e}")

ARCHIVO_EXCEL = 'HORARIOS_SIMULADOS_CORREGIDO.xlsx'

try:
    df_horarios = pd.read_excel(ARCHIVO_EXCEL, sheet_name='BASE')
    print("[Exito] Excel de horarios cargado correctamente.")
except Exception as e:
    print(f"[Error] Al cargar el Excel: {e}")
    df_horarios = pd.DataFrame()

try:
    df_competencias = pd.read_excel(ARCHIVO_EXCEL, sheet_name='OPTATIVOS COMPETENCIAS GLOBALES')
    df_competencias = df_competencias.rename(columns={"Sigla": "Sigla Asignatura", "Asignatura": "Nombre Asignatura"})
    print("[Exito] Hoja de Competencias Globales cargada correctamente.")
except Exception as e:
    print(f"[Aviso] No se pudo cargar la hoja de Competencias Globales: {e}")
    df_competencias = pd.DataFrame()

[Exito] API key y modelo IA funcionando correctamente
[Exito] Excel de horarios cargado correctamente.
[Exito] Hoja de Competencias Globales cargada correctamente.


# 3. PREPARACIÓN DEL EXCEL Y HERRAMIENTAS DE NORMALIZACIÓN

Convierte los horarios escritos como texto (`Lu 08:30:00 - 10:00:00`) en columnas numéricas (`Dia`, `Inicio`, `Fin`) y define las herramientas de normalización de texto, catálogo de siglas y carreras válidas.

In [3]:
PATRON_HORARIO = re.compile(r"^(Lu|Ma|Mi|Ju|Vi|Sa|Do)\s+(\d{1,2}):(\d{2}):\d{2}\s*-\s*(\d{1,2}):(\d{2}):\d{2}$")
PATRON_SIGLA = re.compile(r"^[A-Z]{3}\d{4}$")
DIAS_SEMANA = ["Lu", "Ma", "Mi", "Ju", "Vi", "Sa"]

CARRERAS_EXCEL = {
    "Mecánica": ["ING. EN MECÁNICA AUTOM. Y AUTOTRÓNICA"],
    "Finanzas": ["ING. EN ADMINISTRACIÓN MENCIÓN FINANZAS", "INGENIERÍA EN ADMINIST. MENCIÓN FINANZAS"],
    "Automatización": ["INGENIERÍA EN INFORMÁTICA"],
    "Ciencia de Datos": ["INGENIERÍA EN INFORMÁTICA"],
    "Desarrollo": ["INGENIERÍA EN INFORMÁTICA"],
    "Ingeniería de Datos": ["INGENIERÍA EN INFORMÁTICA"],
    "Inteligencia Artificial": ["INGENIERÍA EN INFORMÁTICA"]
}

NOMBRES_FORMALES_CARRERA = {
    "Mecánica": "Ingeniería en Mecánica Automotriz y Autotrónica",
    "Finanzas": "Ingeniería en Administración mención Finanzas",
    "Automatización": "Ingeniería en Informática mención Automatización",
    "Ciencia de Datos": "Ingeniería en Informática mención Ciencia de Datos",
    "Desarrollo": "Ingeniería en Informática mención Desarrollo de Software",
    "Ingeniería de Datos": "Ingeniería en Informática mención Ingeniería de Datos",
    "Inteligencia Artificial": "Ingeniería en Informática mención Inteligencia Artificial"
}

CARRERAS_VALIDAS = list(CARRERAS_EXCEL.keys())

def quitar_acentos(texto):
    return "".join(c for c in unicodedata.normalize("NFD", str(texto)) if unicodedata.category(c) != "Mn")

def normalizar(texto):
    return re.sub(r"\s+", " ", quitar_acentos(texto)).strip().lower()

def parsear_horario(texto):
    coincidencia = PATRON_HORARIO.match(str(texto).strip())
    if not coincidencia:
        return (None, None, None)
    dia, h1, m1, h2, m2 = coincidencia.groups()
    inicio = int(h1) * 60 + int(m1)
    fin = int(h2) * 60 + int(m2)
    if fin <= inicio:
        return (None, None, None)
    return (dia, inicio, fin)

def preparar_horarios(df):
    """Agrega las columnas Dia, Inicio y Fin (en minutos) a una tabla de horarios."""
    if df.empty:
        return df
    tripletas = df["Horario"].map(parsear_horario)
    df["Dia"] = [t[0] for t in tripletas]
    df["Inicio"] = [t[1] for t in tripletas]
    df["Fin"] = [t[2] for t in tripletas]
    return df

df_horarios = preparar_horarios(df_horarios)
df_competencias = preparar_horarios(df_competencias)

catalogo_siglas = {}
if not df_horarios.empty:
    catalogo_siglas = df_horarios.drop_duplicates("Sigla Asignatura").set_index("Sigla Asignatura")["Nombre Asignatura"].to_dict()
    print(f"[Exito] {len(df_horarios)} filas del Excel preparadas, {df_horarios['Dia'].notna().sum()} con horario fijo.")

[Exito] 7704 filas del Excel preparadas, 7704 con horario fijo.


# 4. REGISTRO DE EVENTOS INSTRUMENTADO (LOGS, LATENCIA Y TOKENS)

**¿Qué hace esta celda?**
Crea la bitácora instrumentada del sistema. Cada entrada guarda:
- `fecha`, `nivel` (`INFO`, `WARNING`, `ERROR`), `origen`, `mensaje`
- `duracion_ms`: tiempo medido con `time.perf_counter()` en milisegundos para cada etapa (`Agente 1`, `Motor`, `Agente 2`, `Total`, y lectura OCR de malla subida)
- `tokens`: cantidad de tokens consumidos (`usage.total_tokens`) en cada llamada al modelo de lenguaje
- `solicitud_id` y `etapa`: identificadores para reconstruir la trazabilidad completa de cada solicitud y calcular métricas de rendimiento, errores y respaldo.

In [4]:
import logging
import threading
from datetime import datetime

REGISTRO_LOGS = []
REGISTRO_SOLICITUDES = []
CANDADO_LOGS = threading.Lock()
ARCHIVO_LOGS = "registro_logs.log"
_CONTADOR_SOLICITUDES = [0]


def siguiente_solicitud_id():
    with CANDADO_LOGS:
        _CONTADOR_SOLICITUDES[0] += 1
        return _CONTADOR_SOLICITUDES[0]


class ManejadorBitacora(logging.Handler):
    def emit(self, record):
        dur = getattr(record, "duracion_ms", None)
        tok = getattr(record, "tokens", None)
        entrada = {
            "fecha": datetime.fromtimestamp(record.created).strftime("%Y-%m-%d %H:%M:%S"),
            "nivel": record.levelname,
            "origen": getattr(record, "origen", "Sistema"),
            "etapa": getattr(record, "etapa", None),
            "solicitud_id": getattr(record, "solicitud_id", None),
            "duracion_ms": round(float(dur), 2) if dur is not None else None,
            "tokens": int(tok) if tok is not None else 0,
            "mensaje": record.getMessage()
        }
        with CANDADO_LOGS:
            REGISTRO_LOGS.append(entrada)


class FormateadorBitacora(logging.Formatter):
    def format(self, record):
        base = super().format(record)
        dur = getattr(record, "duracion_ms", None)
        tok = getattr(record, "tokens", None)
        sid = getattr(record, "solicitud_id", None)
        partes = []
        if sid is not None:
            partes.append(f"req={sid}")
        if dur is not None:
            partes.append(f"duracion_ms={round(float(dur), 2)}")
        if tok is not None:
            partes.append(f"tokens={int(tok)}")
        return f"{base} | {' | '.join(partes)}" if partes else base


logger_bitacora = logging.getLogger("asistente_horarios")
logger_bitacora.setLevel(logging.INFO)
logger_bitacora.propagate = False
for manejador_previo in list(logger_bitacora.handlers):
    logger_bitacora.removeHandler(manejador_previo)
    manejador_previo.close()

logger_bitacora.addHandler(ManejadorBitacora())
manejador_archivo = logging.FileHandler(ARCHIVO_LOGS, encoding="utf-8")
manejador_archivo.setFormatter(FormateadorBitacora("%(asctime)s | %(levelname)s | %(origen)s | %(message)s"))
logger_bitacora.addHandler(manejador_archivo)


def registrar(nivel, mensaje, origen="Sistema", duracion_ms=None, tokens=None, solicitud_id=None, etapa=None):
    try:
        logger_bitacora.log(
            getattr(logging, nivel),
            mensaje,
            extra={
                "origen": origen,
                "duracion_ms": duracion_ms,
                "tokens": tokens,
                "solicitud_id": solicitud_id,
                "etapa": etapa
            }
        )
    except Exception as e:
        print(f"[Aviso] No se pudo registrar el evento: {e}")


def calcular_metricas_logs():
    """Calcula frecuencia de errores, tasa de resumen de respaldo, latencias y tokens."""
    with CANDADO_LOGS:
        logs = list(REGISTRO_LOGS)
        reqs = list(REGISTRO_SOLICITUDES)

    solicitudes_totales = len(reqs) if reqs else sum(1 for e in logs if e.get("etapa") == "Total")
    errores_totales = sum(1 for e in logs if e["nivel"] == "ERROR")
    warnings_totales = sum(1 for e in logs if e["nivel"] == "WARNING")
    frecuencia_errores = (errores_totales / solicitudes_totales) if solicitudes_totales > 0 else 0.0

    llamadas_a2 = sum(1 for e in logs if e.get("etapa") == "Agente 2" and "Etapa Agente 2" in e["mensaje"])
    usos_respaldo = sum(1 for e in logs if "resumen de respaldo" in e["mensaje"].lower())
    tasa_respaldo_a2 = (usos_respaldo / llamadas_a2) if llamadas_a2 > 0 else 0.0
    tasa_respaldo_total = (usos_respaldo / solicitudes_totales) if solicitudes_totales > 0 else 0.0

    etapas_lat = {}
    for et in ["Agente 1", "Motor", "Agente 2", "Total"]:
        vals = [e["duracion_ms"] for e in logs if e.get("etapa") == et and e.get("duracion_ms") is not None]
        etapas_lat[et] = {
            "n": len(vals),
            "promedio_ms": round(float(np.mean(vals)), 2) if vals else 0.0,
            "mediana_ms": round(float(np.median(vals)), 2) if vals else 0.0,
            "p95_ms": round(float(np.percentile(vals, 95)), 2) if vals else 0.0,
            "total_ms": round(float(np.sum(vals)), 2) if vals else 0.0,
        }

    tokens_a1 = sum(e.get("tokens", 0) or 0 for e in logs if e.get("etapa") == "Agente 1")
    tokens_a2 = sum(e.get("tokens", 0) or 0 for e in logs if e.get("etapa") == "Agente 2")
    tokens_totales = tokens_a1 + tokens_a2

    return {
        "solicitudes_totales": solicitudes_totales,
        "errores_totales": errores_totales,
        "warnings_totales": warnings_totales,
        "frecuencia_errores": round(frecuencia_errores, 4),
        "llamadas_agente2": llamadas_a2,
        "usos_resumen_respaldo": usos_respaldo,
        "tasa_resumen_respaldo": round(tasa_respaldo_a2, 4),
        "tasa_resumen_respaldo_sobre_total": round(tasa_respaldo_total, 4),
        "latencia_etapas": etapas_lat,
        "tokens_agente1": tokens_a1,
        "tokens_agente2": tokens_a2,
        "tokens_totales": tokens_totales,
    }


def exportar_logs(ruta_json="registro_logs_exportado.json", ruta_csv="registro_logs_exportado.csv"):
    with CANDADO_LOGS:
        logs = list(REGISTRO_LOGS)
    with open(ruta_json, "w", encoding="utf-8") as f:
        json.dump(logs, f, ensure_ascii=False, indent=2)
    pd.DataFrame(logs).to_csv(ruta_csv, index=False, encoding="utf-8")
    return ruta_json, ruta_csv


registrar("INFO", "Sistema de logs instrumentado iniciado", "Sistema")
print(f"[Exito] Sistema de logs instrumentado listo ({ARCHIVO_LOGS})")

[Exito] Sistema de logs instrumentado listo (registro_logs.log)


# 5. LECTURA DE LAS MALLAS (VISIÓN + OCR EN ORDEN ORIGINAL)

Mantiene exactamente el mismo orden de análisis implementado originalmente:
1. `detectar_columnas(bgr)` -> localiza la barra amarilla y columnas de semestres.
2. `detectar_tarjetas(bgr, y_hdr)` -> detecta las tarjetas blancas ordenadas de arriba a abajo y de izquierda a derecha `(y, x)`.
3. Por cada tarjeta: `_ocr_texto_tarjeta` -> `_separar_nombre` -> `_ocr_codigo` -> `corregir_sigla`.
4. Validación de siglas repetidas y ordenamiento final por semestre.

Además, incorpora soporte opcional de reporte de progreso (`progress_callback`) para alimentar la barra de carga en Gradio, detección automática de la carrera desde el encabezado de la malla y validación para rechazar imágenes que no correspondan a una malla curricular.

In [5]:
LANG_OCR = "spa"

PATRON_SIGLA_MALLA = re.compile(r"[A-Z]{3}\d{4}")
CATEGORIAS = {
    "especialidades": "Especialidad",
    "formacion sello": "Formación Sello",
    "formacion general": "Formación General",
    "formacion optativa": "Formación Optativa",
}


def _norm(t):
    return normalizar(t)


def detectar_columnas(bgr):
    """Las 8 columnas de semestre se detectan por la barra amarilla bajo cada encabezado."""
    b, g, r = bgr[:, :, 0].astype(int), bgr[:, :, 1].astype(int), bgr[:, :, 2].astype(int)
    amarillo = (r > 220) & (g > 150) & (g < 215) & (b < 90)
    alto_busqueda = min(bgr.shape[0], 1200)
    filas = np.where(amarillo[:alto_busqueda].sum(axis=1) > bgr.shape[1] * 0.4)[0]
    if len(filas) == 0:
        return [], 0
    y = int(np.median(filas))
    linea = amarillo[max(0, y - 2):y + 3].any(axis=0)
    xs = np.where(linea)[0]
    if len(xs) == 0:
        return [], 0
    corridas, ini = [], xs[0]
    for a, b_ in zip(xs, xs[1:]):
        if b_ - a > 12:
            corridas.append((ini, a))
            ini = b_
    corridas.append((ini, xs[-1]))
    corridas = [c for c in corridas if c[1] - c[0] > bgr.shape[1] * 0.05]
    return corridas, int(filas.max())


def detectar_tarjetas(bgr, y_min):
    blanco = ((bgr[:, :, 0] >= 254) & (bgr[:, :, 1] >= 254) & (bgr[:, :, 2] >= 254)).astype(np.uint8) * 255
    blanco[:y_min + 5, :] = 0
    cerrado = cv2.morphologyEx(blanco, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5)))
    contornos, _ = cv2.findContours(cerrado, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    tarjetas = []
    for c in contornos:
        x, y, w, h = cv2.boundingRect(c)
        if w >= bgr.shape[1] * 0.07 and h >= bgr.shape[0] * 0.025 and w * h > 40000:
            tarjetas.append((x, y, w, h))
    return sorted(tarjetas, key=lambda t: (t[1], t[0]))


A2D = {"O": "0", "Q": "0", "I": "1", "L": "1", "S": "5", "B": "8", "Z": "2", "G": "6"}
D2A = {"0": "O", "1": "I", "5": "S", "8": "B", "2": "Z", "6": "G"}


def _forzar_formato(s7):
    """3 letras + 4 digitos: corrige confusiones tipicas I/1, O/0, S/5, B/8, Z/2, G/6."""
    if len(s7) != 7:
        return None
    r = "".join((D2A.get(c, c) if i < 3 else A2D.get(c, c)) for i, c in enumerate(s7))
    return r if PATRON_SIGLA_MALLA.fullmatch(r) else None


def candidatos_codigo(txt):
    s = re.sub(r"[^A-Z0-9]", "", txt.upper())
    crudos = set()
    if len(s) == 7:
        crudos.add(s)
    elif len(s) == 8:
        crudos |= {s[:i] + s[i + 1:] for i in range(8)}
    elif len(s) > 8:
        crudos |= {s[i:i + 7] for i in range(len(s) - 6)}
    return [c for c in (_forzar_formato(x) for x in crudos) if c]


def _ocr_codigo(bgr, caja, catalogo=None):
    x, y, w, h = caja
    k = bgr.shape[1] / 2768.0
    banda = bgr[y + int(5 * k):y + int(52 * k), x + int(4 * k):x + int(w * 0.66)]
    if banda.size == 0:
        return None, 0.0
    gris = cv2.cvtColor(banda, cv2.COLOR_BGR2GRAY)
    votos = []
    for escala in (3, 4):
        g = cv2.resize(gris, None, fx=escala, fy=escala, interpolation=cv2.INTER_CUBIC)
        for umbral in (170, 200, 225):
            _, bn = cv2.threshold(g, umbral, 255, cv2.THRESH_BINARY)
            bn = cv2.copyMakeBorder(bn, 20, 20, 20, 20, cv2.BORDER_CONSTANT, value=255)
            txt = pytesseract.image_to_string(bn, lang=LANG_OCR, config="--psm 7")
            votos.extend(candidatos_codigo(txt))
    if not votos:
        return None, 0.0
    conteo = {v: votos.count(v) for v in set(votos)}
    mejor = max(conteo.values())
    empatados = [v for v, c in conteo.items() if c == mejor]
    ganador = next((v for v in empatados if catalogo and v in catalogo), empatados[0])
    return ganador, min(1.0, conteo[ganador] / 6.0)


def _ocr_texto_tarjeta(bgr, caja):
    x, y, w, h = caja
    recorte = bgr[y + 3:y + h - 3, x + 3:x + w - 3]
    if recorte.size == 0:
        return []
    gris = cv2.cvtColor(recorte, cv2.COLOR_BGR2GRAY)
    g = cv2.resize(gris, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
    datos = pytesseract.image_to_data(g, lang=LANG_OCR, config="--psm 6", output_type=pytesseract.Output.DICT)
    palabras = []
    for i, t in enumerate(datos["text"]):
        t = t.strip()
        if t and float(datos["conf"][i]) > -1:
            palabras.append((t, datos["left"][i], datos["top"][i], datos["height"][i]))
    lineas = []
    for p in sorted(palabras, key=lambda p: (p[2], p[1])):
        if lineas and abs(p[2] - lineas[-1][0][2]) < max(12, p[3] * 0.6):
            lineas[-1].append(p)
        else:
            lineas.append([p])
    return [" ".join(q[0] for q in sorted(l, key=lambda q: q[1])) for l in lineas]


ETIQUETAS = ["especialidades", "formacion sello", "formacion general", "formacion optativa"]


def _es_etiqueta(n):
    """True si la linea es la etiqueta de categoria (con o sin '5 semestres'), tolerando ruido de OCR."""
    limpio = re.sub(r"\b\d\s*semestres?\b", "", n).strip()
    if not limpio:
        return None
    for et in ETIQUETAS:
        if difflib.SequenceMatcher(None, limpio, et).ratio() >= 0.82:
            return et
    return None


def _limpiar_ruido_inicial(linea):
    """Quita iconos mal leidos al inicio ('i', '§', '()', 'J'...)."""
    toks = linea.split()
    while len(toks) > 1 and not re.search(r"[A-Za-z]{3,}", toks[0]) and _norm(toks[0]) not in {"de", "en", "el", "la", "y", "a"}:
        toks.pop(0)
    return " ".join(toks)


def _limpiar_nombre(nombre):
    """Corrige numeros romanos mal leidos ('|', 'l') y quita caracteres que romperian tablas markdown."""
    nombre = re.sub(r"\|+", lambda m: "I" * len(m.group()), nombre)
    toks = nombre.split()
    if len(toks) > 1 and re.fullmatch(r"[Il]{1,3}", toks[-1]):
        toks[-1] = "I" * len(toks[-1])
    nombre = " ".join(toks)
    return re.sub(r"[^\w\s().,&/\-ÁÉÍÓÚÜÑáéíóúüñ]", "", nombre).strip()


def _separar_nombre(lineas):
    """Devuelve (nombre, categoria, semestres_span, es_cupo) a partir de las lineas OCR de una tarjeta."""
    lineas = [l.strip() for l in lineas if l.strip()]
    es_cupo, categoria, span = False, None, 1
    restantes = []
    for linea in lineas:
        n = _norm(linea)
        if n.startswith("cupo"):
            es_cupo = True
            resto = re.sub(r"^(cupo\s*electivo|cupo)\s*", "", n)
            if len(resto) < 3:
                continue
            linea = linea[len(linea) - len(resto):] if len(linea) >= len(resto) else linea
        restantes.append(linea)

    while restantes:
        n = _norm(restantes[-1])
        m = re.search(r"(\d)\s*semestres?", n)
        et = _es_etiqueta(n)
        if et:
            categoria = CATEGORIAS.get(et)
            if m:
                span = int(m.group(1))
            restantes.pop()
        elif m and len(n) < 14:
            span = int(m.group(1))
            restantes.pop()
        else:
            break

    if restantes and not es_cupo:
        primera = restantes[0].split()
        if primera and (candidatos_codigo(primera[0]) or (re.fullmatch(r"[A-Z0-9]{5,8}", primera[0]) and len(primera) <= 3)):
            restantes.pop(0)
    restantes = [_limpiar_ruido_inicial(l) for l in restantes]
    return _limpiar_nombre(" ".join(restantes)), categoria, span, es_cupo


def tipo_cupo(nombre):
    n = _norm(nombre)
    if "cristiana" in n:
        return "formacion_cristiana"
    if "competencias globales" in n or "capacidades" in n:
        return "competencias_globales"
    if "disciplinar" in n:
        return "optativo_disciplinar"
    return "otro"


def corregir_sigla(sigla, nombre, catalogo, confianza=1.0):
    """catalogo: {sigla: nombre_excel}.
    Una lectura estable (confianza >= 0.66) NUNCA se reemplaza: solo se marca si existe en el Excel.
    Solo si la lectura es inestable se intenta corregir 1 caracter, y exigiendo nombre casi identico."""
    if sigla is None:
        return None, False
    def sim(a, b):
        return difflib.SequenceMatcher(None, _norm(a), _norm(b)).ratio() if a and b else 0.0
    if confianza >= 0.66 or sigla in catalogo:
        return sigla, sigla in catalogo
    vecinos = [s for s in catalogo if len(s) == len(sigla) and sum(a != b for a, b in zip(s, sigla)) == 1]
    buenos = [s for s in vecinos if nombre and sim(nombre, catalogo[s]) >= 0.85]
    if len(buenos) == 1:
        return buenos[0], True
    return sigla, False


def _identificar_carrera_desde_texto(texto_crudo):
    t = _norm(texto_crudo)
    if not t:
        return None
    reglas = [
        ("Inteligencia Artificial", ["inteligencia artificial"]),
        ("Ciencia de Datos", ["ciencia de datos", "ciencias de datos"]),
        ("Ingeniería de Datos", ["ingenieria de datos"]),
        ("Desarrollo", ["desarrollo de software", "desarrollo"]),
        ("Automatización", ["automatizacion"]),
        ("Mecánica", ["mecanica", "autotronica", "automotriz"]),
        ("Finanzas", ["finanzas", "administracion"])
    ]
    for clave, patrones in reglas:
        if any(p in t for p in patrones):
            return clave
    return None


def detectar_carrera_malla(bgr, y_hdr, semestres_dict, nombre_archivo=""):
    """Detecta la carrera de la malla subida usando OCR en el encabezado superior,
    con respaldo en el cruce de siglas verificadas contra el Excel."""
    alto_banner = max(y_hdr, int(bgr.shape[0] * 0.14))
    banner = bgr[0:min(bgr.shape[0], alto_banner), :]
    gris = cv2.cvtColor(banner, cv2.COLOR_BGR2GRAY)
    txt_banner = pytesseract.image_to_string(gris, lang=LANG_OCR, config="--psm 6")
    carrera = _identificar_carrera_desde_texto(txt_banner)
    if carrera:
        return carrera

    # Segundo intento: binarizacion invertida (texto blanco sobre fondo oscuro en encabezado Duoc)
    _, bn_inv = cv2.threshold(gris, 150, 255, cv2.THRESH_BINARY_INV)
    txt_banner_inv = pytesseract.image_to_string(bn_inv, lang=LANG_OCR, config="--psm 6")
    carrera = _identificar_carrera_desde_texto(txt_banner_inv)
    if carrera:
        return carrera

    # Tercer intento: cruce de siglas verificadas con df_horarios
    if not df_horarios.empty and semestres_dict:
        siglas_malla = {
            it["sigla"]
            for items in semestres_dict.values()
            for it in items
            if it.get("tipo") == "asignatura" and it.get("sigla")
        }
        if siglas_malla:
            coincidencias = df_horarios[df_horarios["Sigla Asignatura"].isin(siglas_malla)]
            if not coincidencias.empty:
                carrera_excel_top = coincidencias["Carrera"].value_counts().idxmax()
                if "MECÁNICA" in carrera_excel_top.upper():
                    return "Mecánica"
                if "FINANZAS" in carrera_excel_top.upper():
                    return "Finanzas"
                if "INFORMÁTICA" in carrera_excel_top.upper():
                    # Revisar nombres de asignaturas de especialidad para distinguir mencion
                    nombres_unidos = " ".join(
                        _norm(it.get("nombre", ""))
                        for items in semestres_dict.values()
                        for it in items
                    )
                    c_mencion = _identificar_carrera_desde_texto(nombres_unidos)
                    if c_mencion:
                        return c_mencion
                    return _identificar_carrera_desde_texto(nombre_archivo) or "Desarrollo"

    return _identificar_carrera_desde_texto(nombre_archivo)


def leer_malla(ruta, catalogo=None, progress_callback=None, nombre_archivo=""):
    """Lee una imagen de malla curricular siguiendo exactamente el orden original de analisis:
    1) detectar_columnas -> 2) detectar_tarjetas -> 3) OCR tarjeta por tarjeta -> 4) validar duplicados y ordenar."""
    catalogo = catalogo or {}
    if progress_callback:
        progress_callback(0.10, "Cargando imagen y buscando estructura de columnas...")

    bgr = cv2.imdecode(np.fromfile(ruta, dtype=np.uint8), cv2.IMREAD_COLOR)
    if bgr is None:
        return {
            "es_malla_valida": False,
            "motivo_invalida": "El archivo no pudo decodificarse como una imagen valida (.JPG o .PNG).",
            "semestres": {},
            "electivos": False,
            "advertencias": ["Archivo de imagen corrupto o formato no soportado"]
        }

    columnas, y_hdr = detectar_columnas(bgr)
    if not columnas or len(columnas) < 4:
        return {
            "es_malla_valida": False,
            "motivo_invalida": "La imagen subida no es una malla curricular (no se detectaron las columnas de semestres).",
            "semestres": {},
            "electivos": False,
            "advertencias": ["No se detectaron columnas de semestre"]
        }

    if progress_callback:
        progress_callback(0.22, f"Detectando tarjetas de asignaturas ({len(columnas)} columnas encontradas)...")

    tarjetas = detectar_tarjetas(bgr, y_hdr)
    if len(tarjetas) < 6:
        return {
            "es_malla_valida": False,
            "motivo_invalida": "La imagen subida no es una malla curricular (no contiene tarjetas de asignaturas).",
            "semestres": {},
            "electivos": False,
            "advertencias": ["Cantidad insuficiente de tarjetas detectadas"]
        }

    semestres, advertencias = {}, []
    total_tarjetas = len(tarjetas)

    for idx_t, (x, y, w, h) in enumerate(tarjetas):
        if progress_callback and (idx_t % 3 == 0 or idx_t == total_tarjetas - 1):
            frac = 0.25 + 0.60 * ((idx_t + 1) / total_tarjetas)
            progress_callback(frac, f"Analizando tarjetas con OCR ({idx_t + 1}/{total_tarjetas})...")

        izq, der = x, x + w
        solapes = [max(0, min(der, c1) - max(izq, c0)) / (c1 - c0) for c0, c1 in columnas]
        cubiertas = [i for i, s in enumerate(solapes) if s > 0.35]
        if not cubiertas:
            continue
        inicio = cubiertas[0]
        lineas = _ocr_texto_tarjeta(bgr, (x, y, w, h))
        nombre, categoria, span, es_cupo = _separar_nombre(lineas)

        span = max(span, len(cubiertas))
        sigla, conf = (None, 0.0) if es_cupo else _ocr_codigo(bgr, (x, y, w, h), catalogo)
        if es_cupo or (sigla is None and any(_norm(l).startswith(("electivo", "optativo")) for l in lineas[:3])):
            if not nombre:
                advertencias.append(f"Tarjeta de cupo sin nombre legible en x={x}, y={y}")
                continue
            item = {"tipo": "cupo", "sigla": None, "nombre": nombre, "categoria": categoria,
                    "semestres_span": span, "clase_cupo": tipo_cupo(nombre), "verificada": True}
        else:
            sigla, ok = corregir_sigla(sigla, nombre, catalogo, conf)
            if sigla is None:
                advertencias.append(f"Codigo ilegible en la tarjeta '{nombre}' (semestre {inicio + 1}); se buscara por nombre")
            item = {"tipo": "asignatura", "sigla": sigla, "nombre": nombre, "categoria": categoria,
                    "semestres_span": span, "verificada": ok, "confianza_ocr": round(conf, 2)}
        semestres.setdefault(inicio + 1, []).append((y, x, item))

    vistos = {}
    for n, lst in semestres.items():
        for _, _, it in lst:
            if it["sigla"]:
                vistos.setdefault(it["sigla"], []).append(n)
    for sg, ns in vistos.items():
        if len(ns) > 1:
            advertencias.append(f"Codigo {sg} repetido en los semestres {ns}: probable error de lectura, revisar")

    resultado = {}
    for n in sorted(semestres):
        resultado[n] = [it for _, _, it in sorted(semestres[n], key=lambda t: (t[0], t[1]))]

    n_asig = sum(1 for v in resultado.values() for i in v if i["tipo"] == "asignatura")
    n_verificadas = sum(1 for v in resultado.values() for i in v if i["tipo"] == "asignatura" and i.get("verificada"))

    if progress_callback:
        progress_callback(0.90, "Identificando carrera y validando resultados...")

    carrera_detectada = detectar_carrera_malla(bgr, y_hdr, resultado, nombre_archivo=nombre_archivo)

    # Validar que realmente sea una malla curricular (minimo de asignaturas y coincidencias)
    if n_asig < 8 or (catalogo and n_verificadas < 3) or carrera_detectada is None:
        return {
            "es_malla_valida": False,
            "motivo_invalida": "La imagen subida no es una malla curricular valida de Duoc UC.",
            "semestres": {},
            "electivos": False,
            "advertencias": advertencias + ["Validacion estructural: la imagen no corresponde a una malla curricular reconocida"]
        }

    if progress_callback:
        progress_callback(1.0, f"Malla detectada: {carrera_detectada}")

    return {
        "es_malla_valida": True,
        "carrera_detectada": carrera_detectada,
        "nombre_formal": NOMBRES_FORMALES_CARRERA.get(carrera_detectada, carrera_detectada),
        "semestres": resultado,
        "electivos": any(i["tipo"] == "cupo" for v in resultado.values() for i in v),
        "advertencias": advertencias,
        "n_columnas": len(columnas)
    }

# 6. GESTOR DINÁMICO DE MALLA ADJUNTA

**¿Qué cambia respecto a la versión anterior?**
- Ya **no** se requiere subir las 7 imágenes al entorno de Colab ni ejecutar un bucle masivo inicial.
- El diccionario `base_mallas_estructura` inicia vacío y se alimenta dinámicamente cuando el estudiante adjunta su malla (`.JPG` o `.PNG`) desde la interfaz web.
- Incluye validación estricta de formato (`.jpg`, `.jpeg`, `.png`), barra de carga (`gr.Progress`) durante el análisis OCR, rechazo explícito si la foto subida no es una malla, y soporte para borrar la malla analizada al posar el mouse sobre el archivo y presionar `×`.

In [6]:
EXTENSIONES_PERMITIDAS = {".jpg", ".jpeg", ".png"}
base_mallas_estructura = {}


def html_tarjeta_malla_vacia():
    return (
        "<div class='estado-malla-card estado-malla-pendiente'>"
        "<div class='estado-malla-titulo'>Sin malla curricular adjunta</div>"
        "<div class='estado-malla-detalle'>Sube una imagen <b>.JPG</b> o <b>.PNG</b> de tu malla para detectar tu carrera automaticamente.</div>"
        "</div>"
    )


def html_tarjeta_malla_error(mensaje_error):
    return (
        "<div class='estado-malla-card estado-malla-error'>"
        "<div class='estado-malla-titulo'>Archivo rechazado</div>"
        f"<div class='estado-malla-detalle'>{mensaje_error}</div>"
        "</div>"
    )


def html_tarjeta_malla_ok(carrera, nombre_formal, n_asig, n_cupos, n_semestres, dur_ms):
    return (
        "<div class='estado-malla-card estado-malla-ok'>"
        f"<div class='estado-malla-titulo'>Malla detectada: {carrera}</div>"
        f"<div class='estado-malla-subtitulo'>{nombre_formal}</div>"
        f"<div class='estado-malla-detalle'>{n_asig} asignaturas y {n_cupos} cupos electivos en {n_semestres} semestres ({dur_ms:.0f} ms)</div>"
        "<div class='estado-malla-tip'>Posa el mouse sobre el archivo arriba y haz clic en <b>×</b> si deseas borrarlo y subir otro.</div>"
        "</div>"
    )


def procesar_subida_malla(archivo, estado_actual, progress=gr.Progress()):
    """Valida el archivo adjunto, muestra barra de carga durante el OCR y detecta la carrera."""
    estado = dict(estado_actual or {"carrera": None, "semestre": None, "jornada": None, "malla_cargada": False})

    if archivo is None:
        return None, html_tarjeta_malla_vacia(), estado

    ruta = archivo if isinstance(archivo, str) else getattr(archivo, "name", str(archivo))
    nombre_archivo = os.path.basename(ruta)
    ext = os.path.splitext(nombre_archivo)[1].lower()

    # 1. Validacion estricta de extension (.JPG o .PNG)
    if ext not in EXTENSIONES_PERMITIDAS:
        registrar(
            "WARNING",
            f"Archivo rechazado por formato no permitido ({ext or 'sin extension'}): {nombre_archivo}",
            "Lectura de mallas"
        )
        estado["carrera"] = None
        estado["malla_cargada"] = False
        return (
            None,
            html_tarjeta_malla_error(f"El archivo <b>{nombre_archivo}</b> no es JPG ni PNG. Solo se aceptan imagenes .JPG o .PNG."),
            estado
        )

    # 2. Analisis OCR con barra de carga manteniendo el orden original de lectura
    t0 = time.perf_counter()
    try:
        resultado = leer_malla(
            ruta,
            catalogo=catalogo_siglas,
            progress_callback=lambda val, desc: progress(val, desc=desc),
            nombre_archivo=nombre_archivo
        )
        dur_ms = (time.perf_counter() - t0) * 1000.0
    except Exception as e:
        dur_ms = (time.perf_counter() - t0) * 1000.0
        registrar(
            "ERROR",
            f"Error inesperado al analizar la malla '{nombre_archivo}': {type(e).__name__}: {e}",
            "Lectura de mallas",
            duracion_ms=dur_ms,
            tokens=0
        )
        estado["carrera"] = None
        estado["malla_cargada"] = False
        return (
            None,
            html_tarjeta_malla_error("Ocurrio un error al procesar la imagen. Intenta con otra imagen .JPG o .PNG."),
            estado
        )

    # 3. Validacion cuando la imagen subida NO es una malla curricular
    if not resultado.get("es_malla_valida"):
        motivo = resultado.get("motivo_invalida", "La imagen subida no es una malla curricular.")
        registrar(
            "WARNING",
            f"Imagen rechazada (no es una malla): '{nombre_archivo}' - {motivo}",
            "Lectura de mallas",
            duracion_ms=dur_ms,
            tokens=0
        )
        estado["carrera"] = None
        estado["malla_cargada"] = False
        return (
            None,
            html_tarjeta_malla_error("La imagen subida no es una malla. Adjunta una imagen oficial de tu malla curricular (.JPG o .PNG)."),
            estado
        )

    # 4. Malla valida: guardar en base_mallas_estructura y actualizar memoria
    carrera = resultado["carrera_detectada"]
    base_mallas_estructura[carrera] = resultado
    semestres = resultado["semestres"]
    n_asig = sum(1 for v in semestres.values() for i in v if i["tipo"] == "asignatura")
    n_cupos = sum(1 for v in semestres.values() for i in v if i["tipo"] == "cupo")

    registrar(
        "INFO",
        f"Malla '{nombre_archivo}' analizada -> Carrera detectada: {carrera} ({n_asig} asignaturas, {n_cupos} cupos en {len(semestres)} semestres)",
        "Lectura de mallas",
        duracion_ms=dur_ms,
        tokens=0
    )
    for aviso in resultado.get("advertencias", []):
        registrar("WARNING", f"Malla {carrera}: {aviso}", "Lectura de mallas")

    estado["carrera"] = carrera
    estado["malla_cargada"] = True
    estado["archivo_malla"] = nombre_archivo

    return (
        ruta,
        html_tarjeta_malla_ok(
            carrera,
            resultado["nombre_formal"],
            n_asig,
            n_cupos,
            len(semestres),
            dur_ms
        ),
        estado
    )


def limpiar_malla_subida(estado_actual):
    """Se ejecuta cuando el usuario posa el mouse sobre el archivo analizado y hace clic en borrar (×)."""
    estado = dict(estado_actual or {"carrera": None, "semestre": None, "jornada": None, "malla_cargada": False})
    carrera_previa = estado.get("carrera")
    if carrera_previa and carrera_previa in base_mallas_estructura:
        del base_mallas_estructura[carrera_previa]
    registrar(
        "INFO",
        f"Usuario elimino la malla adjunta (carrera previa: {carrera_previa or 'ninguna'})",
        "Lectura de mallas"
    )
    estado["carrera"] = None
    estado["malla_cargada"] = False
    estado.pop("archivo_malla", None)
    return html_tarjeta_malla_vacia(), estado


print("[Exito] Gestor dinamico de mallas listo (sin necesidad de precargar las 7 imagenes en Colab).")

[Exito] Gestor dinamico de mallas listo (sin necesidad de precargar las 7 imagenes en Colab).


# 7. CREACIÓN DE HERRAMIENTAS DE ARMADO (MOTOR DE HORARIOS)

Motor determinista que cruza la malla curricular cargada con las secciones reales del Excel, maximiza las asignaturas sin topes horarios y acomoda los cupos electivos en los bloques libres.

In [7]:
def normalizar_jornada(valor):
    texto = normalizar(valor or "")
    if texto.startswith("diurn"):
        return "Diurno"
    if texto.startswith("vesper"):
        return "Vespertino"
    return None

def normalizar_semestre(valor):
    if valor is None:
        return None
    texto = normalizar(valor)
    numero = re.search(r"\d+", texto)
    if numero:
        return int(numero.group())
    palabras = {"primer": 1, "segundo": 2, "tercer": 3, "cuarto": 4, "quinto": 5,
                "sexto": 6, "septimo": 7, "octavo": 8, "noveno": 9, "decimo": 10}
    for palabra, valor_numerico in palabras.items():
        if palabra in texto:
            return valor_numerico
    return None

def normalizar_carrera(valor):
    texto = normalizar(valor or "")
    if not texto:
        return None
    alias = {"ia": "Inteligencia Artificial", "software": "Desarrollo", "automotriz": "Mecánica"}
    for clave in CARRERAS_VALIDAS:
        if normalizar(clave) == texto:
            return clave
    for palabra, clave in alias.items():
        if re.search(rf"\b{palabra}\b", texto):
            return clave
    coincidencias = [c for c in CARRERAS_VALIDAS if normalizar(c) in texto]
    if not coincidencias and len(texto) >= 5:
        coincidencias = [c for c in CARRERAS_VALIDAS if texto in normalizar(c)]
    if len(coincidencias) == 1:
        return coincidencias[0]
    return None

def celda(texto):
    """Texto seguro para una celda de tabla markdown."""
    return str(texto).replace("|", "/").replace("\n", " ")

def formatear_hora(minutos):
    return f"{minutos // 60:02d}:{minutos % 60:02d}"

NOMBRE_CUPO = {
    "formacion_cristiana": "Formación Cristiana",
    "competencias_globales": "Capacidades y Competencias Globales",
    "optativo_disciplinar": "Optativo de Formación Disciplinar",
    "otro": "Cupo electivo",
}
PALABRAS_NO_DISCIPLINAR = ["ajedrez", "basquetbol", "voleibol", "futbol", "tenis de mesa", "natacion",
                           "handbol", "atletismo", "entrenamiento comunicacional"]

def construir_secciones(filas):
    secciones = []
    for codigo, grupo in filas.groupby("Sección", sort=True):
        bloques = sorted({(d, int(i), int(f)) for d, i, f in zip(grupo["Dia"], grupo["Inicio"], grupo["Fin"]) if pd.notna(d)})
        docentes = grupo["Docente"].dropna().astype(str).unique().tolist()
        virtual = grupo["ASIGNATURA VIRTUAL SINCRONICA"].notna().any()
        secciones.append({
            "codigo": codigo,
            "sigla": grupo["Sigla Asignatura"].iloc[0],
            "nombre": grupo["Nombre Asignatura"].iloc[0],
            "bloques": bloques,
            "docente": " / ".join(docentes) if docentes else "Sin docente asignado en el Excel",
            "modalidad": "Online sincrona" if virtual else "Presencial"
        })
    return secciones

def opciones_de_cupo(clase, carrera, jornada):
    """Devuelve (secciones_en_la_jornada, jornadas_donde_existe) para un cupo electivo, solo con datos del Excel."""
    nombres_carrera = CARRERAS_EXCEL[carrera]
    if clase == "formacion_cristiana":
        base = df_horarios[df_horarios["Carrera"].isin(nombres_carrera) &
                           df_horarios["Nivel"].astype(str).str.contains("Formación Cristiana", case=False)]
    elif clase == "competencias_globales":
        base = df_competencias
    elif clase == "optativo_disciplinar":
        base = df_horarios[df_horarios["Carrera"].isin(nombres_carrera) & (df_horarios["Nivel"].astype(str) == "Optativos")]
        base = base[~base["Nombre Asignatura"].map(normalizar).str.contains("|".join(PALABRAS_NO_DISCIPLINAR))]
    else:
        return [], []
    if base is None or base.empty:
        return [], []
    en_jornada = base[base["Jornada"].isin([jornada, "Diurno y Vespertino"])]
    return construir_secciones(en_jornada) if not en_jornada.empty else [], sorted(base["Jornada"].unique().tolist())

_cache_planes = {}

def planes_de_la_malla(carrera):
    if carrera not in _cache_planes:
        semestres = base_mallas_estructura.get(carrera, {}).get("semestres", {})
        siglas = {i["sigla"] for v in semestres.values() for i in v if i["tipo"] == "asignatura" and i["sigla"]}
        base = df_horarios[df_horarios["Carrera"].isin(CARRERAS_EXCEL[carrera])]
        _cache_planes[carrera] = set(base[base["Sigla Asignatura"].isin(siglas)]["Plan"].unique())
    return _cache_planes[carrera]

def buscar_materias(carrera, semestre, jornada):
    items = base_mallas_estructura.get(carrera, {}).get("semestres", {}).get(semestre, [])
    base = df_horarios[df_horarios["Carrera"].isin(CARRERAS_EXCEL[carrera])]
    nombres_excel = base["Nombre Asignatura"].map(normalizar)
    materias = []
    for indice, item in enumerate(items):
        if item["tipo"] == "cupo":
            secciones, jornadas = opciones_de_cupo(item["clase_cupo"], carrera, jornada)
            if secciones:
                estado = "cupo_con_opciones"
            elif jornadas:
                estado = "cupo_solo_otra_jornada"
            else:
                estado = "cupo_sin_opciones"
            materias.append({"id": f"CUPO-{indice}", "tipo": "cupo", "sigla": None, "sigla_excel": None,
                             "nombre": item["nombre"], "clase_cupo": item["clase_cupo"],
                             "semestres_span": item.get("semestres_span", 1),
                             "secciones": secciones, "estado": estado, "jornadas_disponibles": jornadas})
            continue

        sigla = item["sigla"]
        filas = base[base["Sigla Asignatura"] == sigla] if sigla else base.iloc[0:0]
        sigla_excel = sigla
        posible_equivalente = None
        if filas.empty and item["nombre"]:
            por_nombre = base[nombres_excel == normalizar(item["nombre"])]
            if por_nombre["Sigla Asignatura"].nunique() == 1:
                posible_equivalente = por_nombre["Sigla Asignatura"].iloc[0]
        nombre = filas["Nombre Asignatura"].iloc[0] if not filas.empty else (item["nombre"] or "Nombre no legible en la malla")
        materia = {"id": sigla or f"SIN-CODIGO-{indice}", "tipo": "asignatura", "sigla": sigla or "(sin codigo)",
                   "sigla_excel": sigla_excel or "(sin codigo)", "nombre": nombre, "secciones": [],
                   "estado": "sin_secciones_en_excel", "jornadas_disponibles": [], "posible_equivalente": posible_equivalente}
        if not filas.empty:
            filas_jornada = filas[filas["Jornada"].isin([jornada, "Diurno y Vespertino"])]
            if filas_jornada.empty:
                materia["estado"] = "solo_en_otra_jornada"
                materia["jornadas_disponibles"] = sorted(filas["Jornada"].unique().tolist())
            else:
                materia["estado"] = "disponible"
                materia["secciones"] = construir_secciones(filas_jornada)
        materias.append(materia)
    return materias

def hay_choque(bloques_a, bloques_b):
    for dia_a, ini_a, fin_a in bloques_a:
        for dia_b, ini_b, fin_b in bloques_b:
            if dia_a == dia_b and ini_a < fin_b and ini_b < fin_a:
                return True
    return False

def armar_horario(materias):
    candidatas = sorted([m for m in materias if m["tipo"] == "asignatura" and m["secciones"]],
                        key=lambda m: len(m["secciones"]))
    mejor = {"cantidad": -1, "seleccion": {}}
    contador = [0]

    def explorar(i, elegidas, ocupados):
        contador[0] += 1
        if contador[0] > 200000:
            return
        if len(elegidas) + (len(candidatas) - i) <= mejor["cantidad"]:
            return
        if i == len(candidatas):
            mejor["cantidad"] = len(elegidas)
            mejor["seleccion"] = dict(elegidas)
            return
        materia = candidatas[i]
        for seccion in materia["secciones"]:
            if not hay_choque(seccion["bloques"], ocupados):
                elegidas[materia["id"]] = seccion
                explorar(i + 1, elegidas, ocupados + seccion["bloques"])
                del elegidas[materia["id"]]
        explorar(i + 1, elegidas, ocupados)

    explorar(0, {}, [])
    seleccion = dict(mejor["seleccion"])

    ocupados = [b for s in seleccion.values() for b in s["bloques"]]
    for cupo in [m for m in materias if m["tipo"] == "cupo" and m["secciones"]]:
        orden = sorted(cupo["secciones"], key=lambda s: (not s["bloques"], s["nombre"], s["codigo"]))
        for seccion in orden:
            if not hay_choque(seccion["bloques"], ocupados):
                seleccion[cupo["id"]] = seccion
                ocupados = ocupados + seccion["bloques"]
                break
    return seleccion

def construir_tabla_horario(seleccion):
    bloques = []
    for _, seccion in seleccion.items():
        for dia, ini, fin in seccion["bloques"]:
            bloques.append((dia, ini, fin, seccion["nombre"], seccion["codigo"]))
    if not bloques:
        return ""
    franjas = sorted({(ini, fin) for _, ini, fin, _, _ in bloques})
    celdas = {}
    for dia, ini, fin, nombre, codigo in bloques:
        celdas[(dia, ini, fin)] = f"{celda(nombre)}<br>{codigo}"
    lineas = [
        "| Modulo Horario | Lunes | Martes | Miercoles | Jueves | Viernes | Sabado |",
        "|---|---|---|---|---|---|---|"
    ]
    for ini, fin in franjas:
        fila = [f"{formatear_hora(ini)} - {formatear_hora(fin)}"]
        for dia in DIAS_SEMANA:
            fila.append(celdas.get((dia, ini, fin), ""))
        lineas.append("| " + " | ".join(fila) + " |")
    return "\n".join(lineas)

def construir_tabla_detalle(seleccion, materias):
    lineas = [
        "| Sigla | Asignatura | Seccion | Docente | Modalidad | Horario |",
        "|---|---|---|---|---|---|"
    ]
    for materia in materias:
        seccion = seleccion.get(materia["id"])
        if seccion:
            if seccion["bloques"]:
                horario = "<br>".join(f"{dia} {formatear_hora(i)} - {formatear_hora(f)}" for dia, i, f in seccion["bloques"])
            else:
                horario = "Sin horario fijo en el Excel"
            etiqueta = " (cupo electivo)" if materia["tipo"] == "cupo" else ""
            lineas.append(f"| {seccion['sigla']} | {celda(seccion['nombre'])}{etiqueta} | {seccion['codigo']} | {celda(seccion['docente'])} | {seccion['modalidad']} | {horario} |")
    return "\n".join(lineas)

def construir_tabla_malla(seleccion, materias, jornada):
    lineas = ["| Tipo | Codigo | Segun la malla | Estado en esta propuesta |", "|---|---|---|---|"]
    for m in materias:
        if m["tipo"] == "asignatura":
            if m["id"] in seleccion:
                estado = f"Con horario, seccion {seleccion[m['id']]['codigo']}"
            elif m["estado"] == "sin_secciones_en_excel":
                estado = "El Excel no publica secciones para esta asignatura"
            elif m["estado"] == "solo_en_otra_jornada":
                estado = f"Solo publicada en jornada {', '.join(m['jornadas_disponibles'])}"
            else:
                estado = "Tiene secciones, pero todas chocan con otras asignaturas"
            lineas.append(f"| Asignatura | {celda(m['sigla'])} | {celda(m['nombre'])} | {estado} |")
        else:
            clase = NOMBRE_CUPO.get(m["clase_cupo"], "Cupo electivo")
            extra = f" (cupo valido por {m['semestres_span']} semestres)" if m["semestres_span"] > 1 else ""
            if m["id"] in seleccion:
                s = seleccion[m["id"]]
                estado = f"Opcion sugerida del Excel: {s['sigla']} {s['nombre']}, seccion {s['codigo']}"
            elif m["estado"] == "cupo_con_opciones":
                estado = "Hay opciones en el Excel, pero todas chocan con el horario armado"
            elif m["estado"] == "cupo_solo_otra_jornada":
                estado = f"Solo hay opciones en jornada {', '.join(m['jornadas_disponibles'])}"
            else:
                estado = "El Excel no publica opciones para este cupo"
            lineas.append(f"| Cupo electivo | - | {celda(m['nombre'])}{extra} [{clase}] | {celda(estado)} |")
    return "\n".join(lineas)

def generar_propuesta(carrera, semestre, jornada):
    materias = buscar_materias(carrera, semestre, jornada)
    seleccion = armar_horario(materias)

    asignaturas = [m for m in materias if m["tipo"] == "asignatura"]
    cupos = [m for m in materias if m["tipo"] == "cupo"]
    asignadas = [m for m in asignaturas if m["id"] in seleccion]
    sin_excel = [m for m in asignaturas if m["estado"] == "sin_secciones_en_excel"]
    otra_jornada = [m for m in asignaturas if m["estado"] == "solo_en_otra_jornada"]
    con_choque = [m for m in asignaturas if m["estado"] == "disponible" and m["id"] not in seleccion]
    cupos_asignados = [m for m in cupos if m["id"] in seleccion]
    cupos_sin_asignar = [m for m in cupos if m["id"] not in seleccion]

    datos_verificados = {
        "carrera": carrera,
        "semestre": semestre,
        "jornada": jornada,
        "total_asignaturas_en_malla": len(asignaturas),
        "total_cupos_electivos_en_malla": len(cupos),
        "asignadas": [{"sigla": m["sigla"], "nombre": m["nombre"], "seccion": seleccion[m["id"]]["codigo"]} for m in asignadas],
        "cupos_electivos_con_opcion_sugerida": [{"cupo": m["nombre"], "sigla": seleccion[m["id"]]["sigla"], "seccion": seleccion[m["id"]]["codigo"]} for m in cupos_asignados],
        "cupos_electivos_sin_opcion": [{"cupo": m["nombre"]} for m in cupos_sin_asignar],
        "sin_secciones_en_excel": [{"sigla": m["sigla"], "nombre": m["nombre"]} for m in sin_excel],
        "solo_en_otra_jornada": [{"sigla": m["sigla"], "nombre": m["nombre"], "jornadas": m["jornadas_disponibles"]} for m in otra_jornada],
        "no_ubicadas_por_choque_de_horario": [{"sigla": m["sigla"], "nombre": m["nombre"]} for m in con_choque],
        "codigos_distintos_entre_malla_y_excel": [{"sigla_malla": m["sigla"], "sigla_excel": m["sigla_excel"]} for m in asignaturas if m["sigla_excel"] != m["sigla"] and m["estado"] != "sin_secciones_en_excel"]
    }

    observaciones = []
    for m in asignaturas:
        if m["sigla_excel"] != m["sigla"] and m["estado"] != "sin_secciones_en_excel":
            observaciones.append(f"- {m['sigla']} - {m['nombre']}: la malla la registra con el codigo {m['sigla']}, pero el Excel la publica como {m['sigla_excel']}. Se usaron las secciones del Excel porque el nombre coincide exactamente.")

    avisos = []
    for m in sin_excel:
        pista = f" El Excel publica una asignatura con el mismo nombre bajo el codigo {m['posible_equivalente']}; no se uso porque el codigo no coincide con la malla (consultalo con tu coordinacion)." if m.get("posible_equivalente") else ""
        avisos.append(f"- {m['sigla']} - {m['nombre']}: pertenece a la malla de este semestre, pero el Excel no tiene secciones publicadas para ella.{pista}")
    for m in otra_jornada:
        avisos.append(f"- {m['sigla']} - {m['nombre']}: no tiene secciones en jornada {jornada}; el Excel solo la publica en jornada {', '.join(m['jornadas_disponibles'])}.")
    for m in con_choque:
        avisos.append(f"- {m['sigla']} - {m['nombre']}: tiene secciones, pero todas chocan con otras asignaturas ya ubicadas.")
    for m in cupos_sin_asignar:
        if m["estado"] == "cupo_solo_otra_jornada":
            avisos.append(f"- Cupo electivo '{m['nombre']}': el Excel solo publica opciones en jornada {', '.join(m['jornadas_disponibles'])}, no en {jornada}.")
        elif m["estado"] == "cupo_con_opciones":
            avisos.append(f"- Cupo electivo '{m['nombre']}': hay opciones en el Excel, pero todas chocan con el horario armado.")
        else:
            avisos.append(f"- Cupo electivo '{m['nombre']}': el Excel no publica opciones para este cupo; revisalo al momento de inscribir.")

    siglas_validas = {m["sigla"] for m in asignaturas} | {m["sigla_excel"] for m in asignaturas}
    siglas_validas |= {s["sigla"] for m in cupos for s in m["secciones"]}

    return {
        "tabla_horario": construir_tabla_horario(seleccion),
        "tabla_detalle": construir_tabla_detalle(seleccion, materias),
        "tabla_malla": construir_tabla_malla(seleccion, materias, jornada),
        "avisos": avisos,
        "observaciones": observaciones,
        "datos_verificados": datos_verificados,
        "siglas_validas": siglas_validas
    }

# 8. AGENTES CON MEMORIA, REQUISITO DE MALLA E INSTRUMENTACIÓN

**Actualización clave en el Agente 1 (Analista) y el flujo principal:**
- El estado en memoria ahora rastrea `malla_cargada` (`True`/`False`) además de `carrera`, `semestre` y `jornada`.
- Si el estudiante envía una solicitud sin haber adjuntado una malla válida en el panel izquierdo, `agente_analista` captura de todas formas el `semestre` y la `jornada` (y detecta si mencionó otra carrera), pero `procesar_solicitud` exige primero adjuntar la imagen de la malla curricular (`.JPG` o `.PNG`).
- Mantiene la instrumentación completa de `duracion_ms` (`perf_counter()`) y `tokens` (`usage.total_tokens`) por etapa.

In [8]:
def _extraer_total_tokens(respuesta):
    try:
        usage = getattr(respuesta, "usage", None)
        if usage is None:
            return 0
        total = getattr(usage, "total_tokens", None)
        if total is not None:
            return int(total)
        prompt_t = getattr(usage, "prompt_tokens", 0) or 0
        comp_t = getattr(usage, "completion_tokens", 0) or 0
        return int(prompt_t + comp_t)
    except Exception:
        return 0


# Agente 1: Analista que lee el mensaje, verifica requisito de malla y actualiza la memoria de estado
def agente_analista(mensaje_usuario, estado_actual, solicitud_id=None, metricas_out=None):
    t0 = time.perf_counter()
    malla_cargada = bool(estado_actual.get("malla_cargada"))
    prompt = f"""
    Eres un analizador de texto estricto para un sistema academico donde el estudiante DEBE adjuntar su malla curricular en imagen.

    ESTADO ACTUAL EN MEMORIA:
    {json.dumps(estado_actual, ensure_ascii=False)}

    CARRERAS VALIDAS (usa exactamente uno de estos nombres o null):
    {json.dumps(CARRERAS_VALIDAS, ensure_ascii=False)}

    NUEVO MENSAJE DEL USUARIO:
    "{mensaje_usuario}"

    REGLAS:
    1. Busca referencias a la carrera, semestre y jornada en el nuevo mensaje.
    2. "carrera" debe ser exactamente uno de los nombres de CARRERAS VALIDAS o null. Si el ESTADO ACTUAL ya tiene "malla_cargada": true, respeta la "carrera" de la malla cargada salvo que el usuario pida explicitamente cambiar a otra carrera.
    3. "semestre" debe ser un numero entero (ej. tercer semestre = 3, 4to = 4, 3er sem = 3).
    4. "jornada" debe ser exactamente "Diurno" o "Vespertino".
    5. "requiere_malla" debe ser true si "malla_cargada" en el ESTADO ACTUAL es false, o false si ya esta cargada.
    6. Si el usuario pide cambiar semestre o jornada (ej. "y en vespertino?"), reemplazalo. Si no cambia un dato, manten el valor que ya estaba en el ESTADO ACTUAL.
    7. Nunca inventes un dato que el usuario no haya dicho: si no hay informacion, deja null.
    8. Responde UNICAMENTE con un objeto JSON valido con las claves "carrera", "semestre", "jornada" y "requiere_malla". No uses texto adicional ni explicaciones.
    """
    tokens = 0
    try:
        respuesta = cliente_ai.chat.completions.create(
            model=modelo_elegido,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.1
        )
        tokens = _extraer_total_tokens(respuesta)
        contenido = respuesta.choices[0].message.content
        bloque_json = re.search(r"\{.*\}", contenido, re.DOTALL)
        if bloque_json is None:
            raise ValueError("la respuesta del modelo no contiene un JSON")
        crudo = json.loads(bloque_json.group())
    except Exception as e:
        duracion_ms = (time.perf_counter() - t0) * 1000.0
        print("Error en Agente Analista:", e)
        registrar(
            "ERROR",
            f"Fallo del Agente 1 (Analista): {type(e).__name__}: {e}. Se mantiene el estado anterior.",
            "Agente 1",
            duracion_ms=duracion_ms,
            tokens=tokens,
            solicitud_id=solicitud_id,
            etapa="Agente 1"
        )
        if metricas_out is not None:
            metricas_out["duracion_agente1_ms"] = round(duracion_ms, 2)
            metricas_out["tokens_agente1"] = tokens
            metricas_out["tuvo_error"] = True
        return estado_actual

    carrera_nueva = normalizar_carrera(crudo.get("carrera"))
    semestre_nuevo = normalizar_semestre(crudo.get("semestre"))
    jornada_nueva = normalizar_jornada(crudo.get("jornada"))
    for campo, bruto, normalizado in (("carrera", crudo.get("carrera"), carrera_nueva),
                                      ("semestre", crudo.get("semestre"), semestre_nuevo),
                                      ("jornada", crudo.get("jornada"), jornada_nueva)):
        if bruto and not normalizado:
            registrar(
                "WARNING",
                f"El Agente 1 devolvio un valor no reconocido para {campo}: {bruto}",
                "Agente 1",
                solicitud_id=solicitud_id,
                etapa="Agente 1"
            )

    estado_res = dict(estado_actual)
    estado_res["semestre"] = semestre_nuevo or estado_actual.get("semestre")
    estado_res["jornada"] = jornada_nueva or estado_actual.get("jornada")
    estado_res["malla_cargada"] = malla_cargada
    estado_res["carrera_mencionada"] = carrera_nueva

    # Si hay una malla cargada, la carrera oficial es la de la malla subida
    if malla_cargada and estado_actual.get("carrera"):
        estado_res["carrera"] = estado_actual.get("carrera")
    else:
        estado_res["carrera"] = carrera_nueva or estado_actual.get("carrera")

    duracion_ms = (time.perf_counter() - t0) * 1000.0
    registrar(
        "INFO",
        f"Etapa Agente 1 completada | malla_cargada={malla_cargada}, carrera={estado_res['carrera']}, "
        f"semestre={estado_res['semestre']}, jornada={estado_res['jornada']}",
        "Agente 1",
        duracion_ms=duracion_ms,
        tokens=tokens,
        solicitud_id=solicitud_id,
        etapa="Agente 1"
    )
    if metricas_out is not None:
        metricas_out["duracion_agente1_ms"] = round(duracion_ms, 2)
        metricas_out["tokens_agente1"] = tokens
    return estado_res


def validar_texto_agente(texto, siglas_validas, solicitud_id=None):
    sin_tablas = "\n".join(l for l in texto.replace("```", "").splitlines() if not l.strip().startswith("|")).strip()
    for sigla in re.findall(r"[A-Z]{3}\d{4}", sin_tablas):
        if sigla not in siglas_validas:
            registrar(
                "WARNING",
                f"Filtro de seguridad: el Agente 2 menciono la sigla {sigla}, que no existe en los datos verificados. Resumen descartado.",
                "Agente 2",
                solicitud_id=solicitud_id,
                etapa="Agente 2"
            )
            return None
    return sin_tablas or None


# Agente 2: Redactor que resume unicamente los datos verificados
def agente_redactor(propuesta, solicitud_id=None, metricas_out=None):
    t0 = time.perf_counter()
    resumen_base = (f"Propuesta construida solo con la malla y el Excel de horarios para {propuesta['datos_verificados']['carrera']}, "
                    f"semestre {propuesta['datos_verificados']['semestre']}, jornada {propuesta['datos_verificados']['jornada']}.")
    prompt = f"""
    Eres el Asistente Virtual DUOC UC.
    Redacta un resumen breve de la propuesta de horario usando UNICAMENTE los datos verificados.

    DATOS VERIFICADOS:
    {json.dumps(propuesta['datos_verificados'], ensure_ascii=False)}

    REGLAS ESTRICTAS:
    1. Maximo 4 oraciones.
    2. Indica cuantas asignaturas del semestre quedaron con horario asignado de un total de las que tiene la malla.
    3. Si hay asignaturas sin secciones en el Excel, en otra jornada o con choque, mencionalas por su sigla tal como aparecen en los datos y sugiere consultar con la sede o coordinacion.
    4. Prohibido nombrar asignaturas, siglas, secciones, docentes u horarios que no esten en los datos verificados.
    5. No generes tablas. No utilices emojis.
    6. Si hay codigos distintos entre malla y Excel, indicalo en una sola oracion.
    7. Los cupos electivos (Formacion Cristiana, Competencias Globales, Optativos) no son asignaturas fijas: son cupos en los que el estudiante elige una opcion. Nombralos como cupos y di que las opciones indicadas son solo sugerencias tomadas del Excel.
    """
    tokens = 0
    try:
        respuesta = cliente_ai.chat.completions.create(
            model=modelo_elegido,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.2
        )
        tokens = _extraer_total_tokens(respuesta)
        texto = validar_texto_agente(respuesta.choices[0].message.content, propuesta["siglas_validas"], solicitud_id=solicitud_id)
        duracion_ms = (time.perf_counter() - t0) * 1000.0
        if metricas_out is not None:
            metricas_out["duracion_agente2_ms"] = round(duracion_ms, 2)
            metricas_out["tokens_agente2"] = tokens
        if texto:
            registrar(
                "INFO",
                "Etapa Agente 2 completada: resumen redactado y aprobado por el filtro de seguridad",
                "Agente 2",
                duracion_ms=duracion_ms,
                tokens=tokens,
                solicitud_id=solicitud_id,
                etapa="Agente 2"
            )
            return texto
        if metricas_out is not None:
            metricas_out["uso_respaldo"] = True
        registrar(
            "WARNING",
            "Etapa Agente 2 completada: filtro de seguridad activado, se uso el resumen de respaldo",
            "Agente 2",
            duracion_ms=duracion_ms,
            tokens=tokens,
            solicitud_id=solicitud_id,
            etapa="Agente 2"
        )
        return resumen_base
    except Exception as e:
        duracion_ms = (time.perf_counter() - t0) * 1000.0
        print("Error en Agente 2:", e)
        if metricas_out is not None:
            metricas_out["duracion_agente2_ms"] = round(duracion_ms, 2)
            metricas_out["tokens_agente2"] = tokens
            metricas_out["tuvo_error"] = True
            metricas_out["uso_respaldo"] = True
        registrar(
            "ERROR",
            f"Fallo del Agente 2 (Redactor): {type(e).__name__}: {e}. Se usa el resumen de respaldo.",
            "Agente 2",
            duracion_ms=duracion_ms,
            tokens=tokens,
            solicitud_id=solicitud_id,
            etapa="Agente 2"
        )
        return resumen_base


def registrar_resultado_propuesta(propuesta, duracion_ms=None, solicitud_id=None):
    d = propuesta["datos_verificados"]
    registrar(
        "INFO",
        f"Etapa Motor completada: {len(d['asignadas'])} de {d['total_asignaturas_en_malla']} asignaturas con horario y "
        f"{len(d['cupos_electivos_con_opcion_sugerida'])} de {d['total_cupos_electivos_en_malla']} cupos electivos con opcion sugerida",
        "Motor de horarios",
        duracion_ms=duracion_ms,
        tokens=0,
        solicitud_id=solicitud_id,
        etapa="Motor"
    )
    categorias = (("sin_secciones_en_excel", "sin secciones en el Excel"),
                  ("solo_en_otra_jornada", "publicadas solo en otra jornada"),
                  ("no_ubicadas_por_choque_de_horario", "no ubicadas por choque de horario"))
    for clave, texto in categorias:
        if d[clave]:
            siglas = ", ".join(str(x["sigla"]) for x in d[clave])
            registrar(
                "WARNING",
                f"Asignaturas {texto}: {siglas}",
                "Motor de horarios",
                solicitud_id=solicitud_id,
                etapa="Motor"
            )
    if d["cupos_electivos_sin_opcion"]:
        nombres = "; ".join(x["cupo"] for x in d["cupos_electivos_sin_opcion"])
        registrar(
            "WARNING",
            f"Cupos electivos sin opcion disponible: {nombres}",
            "Motor de horarios",
            solicitud_id=solicitud_id,
            etapa="Motor"
        )
    if d["codigos_distintos_entre_malla_y_excel"]:
        pares = ", ".join(f"{x['sigla_malla']}->{x['sigla_excel']}" for x in d["codigos_distintos_entre_malla_y_excel"])
        registrar(
            "INFO",
            f"Codigos distintos entre malla y Excel: {pares}",
            "Motor de horarios",
            solicitud_id=solicitud_id,
            etapa="Motor"
        )


def _finalizar_solicitud(t_total_ini, metricas, estado_final, respuesta_texto):
    dur_total_ms = round((time.perf_counter() - t_total_ini) * 1000.0, 2)
    tokens_total = metricas["tokens_agente1"] + metricas["tokens_agente2"]
    metricas["duracion_total_ms"] = dur_total_ms
    metricas["tokens_total"] = tokens_total
    metricas["estado_final"] = dict(estado_final)

    with CANDADO_LOGS:
        REGISTRO_SOLICITUDES.append(metricas)

    registrar(
        "INFO",
        f"Etapa Total completada (req={metricas['solicitud_id']}) | "
        f"A1={metricas['duracion_agente1_ms']}ms, Motor={metricas['duracion_motor_ms']}ms, A2={metricas['duracion_agente2_ms']}ms",
        "Sistema",
        duracion_ms=dur_total_ms,
        tokens=tokens_total,
        solicitud_id=metricas["solicitud_id"],
        etapa="Total"
    )
    return respuesta_texto, estado_final


def procesar_solicitud(mensaje, estado_actual):
    t_total_ini = time.perf_counter()
    sid = siguiente_solicitud_id()
    estado_actual = dict(estado_actual or {"carrera": None, "semestre": None, "jornada": None, "malla_cargada": False})
    metricas = {
        "solicitud_id": sid,
        "fecha": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "mensaje": str(mensaje),
        "duracion_agente1_ms": 0.0,
        "duracion_motor_ms": 0.0,
        "duracion_agente2_ms": 0.0,
        "duracion_total_ms": 0.0,
        "tokens_agente1": 0,
        "tokens_agente2": 0,
        "tokens_total": 0,
        "tuvo_error": False,
        "uso_respaldo": False,
        "estado_final": dict(estado_actual)
    }

    if not str(mensaje or "").strip():
        registrar(
            "WARNING",
            "Se recibio una solicitud vacia",
            "Sistema",
            duracion_ms=0.0,
            tokens=0,
            solicitud_id=sid,
            etapa="Validacion"
        )
        return _finalizar_solicitud(t_total_ini, metricas, estado_actual, "Por favor, ingresa tu solicitud.")

    registrar("INFO", f"Usuario pidio: {mensaje.strip()[:200]}", "Sistema", solicitud_id=sid)
    nuevo_estado = agente_analista(mensaje, estado_actual, solicitud_id=sid, metricas_out=metricas)

    carrera = nuevo_estado.get("carrera")
    semestre = nuevo_estado.get("semestre")
    jornada = nuevo_estado.get("jornada")
    malla_cargada = bool(nuevo_estado.get("malla_cargada") and carrera and base_mallas_estructura.get(carrera, {}).get("semestres"))

    # 1. Requisito obligatorio: verificar si la imagen de la malla fue adjuntada
    if not malla_cargada:
        registrar("INFO", "Solicitud detenida: falta adjuntar la imagen de la malla curricular", "Sistema", solicitud_id=sid)
        recibidos = []
        if nuevo_estado.get("carrera_mencionada"):
            recibidos.append(f"Carrera mencionada en texto: {nuevo_estado['carrera_mencionada']} (pendiente confirmar con imagen de malla)")
        if semestre:
            recibidos.append(f"Semestre: {semestre}")
        if jornada:
            recibidos.append(f"Jornada: {jornada}")

        faltantes = ["adjuntar la imagen de tu malla curricular (.JPG o .PNG) en el panel izquierdo"]
        if not semestre:
            faltantes.append("el semestre a cursar")
        if not jornada:
            faltantes.append("tu jornada (Diurna o Vespertina)")

        msg = "Datos registrados en memoria hasta el momento:\n"
        if recibidos:
            for item in recibidos:
                msg += f"- {item}\n"
            msg += "\n"
        else:
            msg += "- Ningun dato registrado aun.\n\n"

        msg += f"Para generar tu boletin academico, por favor completa lo siguiente: {', '.join(faltantes)}."
        return _finalizar_solicitud(t_total_ini, metricas, nuevo_estado, msg)

    # 2. Verificar si el usuario menciono una carrera distinta a la malla adjunta
    carrera_mencionada = nuevo_estado.get("carrera_mencionada")
    if carrera_mencionada and carrera_mencionada != carrera:
        registrar(
            "WARNING",
            f"Usuario menciono '{carrera_mencionada}' pero tiene adjunta la malla de '{carrera}'",
            "Sistema",
            solicitud_id=sid
        )
        msg = (
            f"Actualmente tienes adjunta la malla de **{carrera}**, pero mencionaste **{carrera_mencionada}** en tu mensaje.\n\n"
            "Si deseas consultar otra carrera, posa el mouse sobre el archivo adjunto en el panel izquierdo, "
            "haz clic en **×** para borrarlo y sube la malla correspondiente."
        )
        return _finalizar_solicitud(t_total_ini, metricas, nuevo_estado, msg)

    # 3. Validar semestre dentro de la malla cargada
    if carrera and semestre and semestre not in base_mallas_estructura.get(carrera, {}).get("semestres", {}):
        disponibles = sorted(base_mallas_estructura.get(carrera, {}).get("semestres", {}).keys())
        nuevo_estado["semestre"] = None
        registrar(
            "WARNING",
            f"La malla de {carrera} no tiene un semestre {semestre} registrado",
            "Sistema",
            solicitud_id=sid
        )
        msg = f"La malla de {carrera} no tiene un semestre {semestre} registrado."
        if disponibles:
            msg += f" Semestres disponibles: {', '.join(str(s) for s in disponibles)}."
        return _finalizar_solicitud(t_total_ini, metricas, nuevo_estado, msg + " Por favor, indicame un semestre valido.")

    # 4. Verificar si falta semestre o jornada
    faltantes = []
    if not semestre: faltantes.append("el semestre a cursar")
    if not jornada: faltantes.append("tu jornada (Diurna o Vespertina)")

    if faltantes:
        registrar("INFO", f"Datos incompletos, falta: {', '.join(faltantes)}", "Sistema", solicitud_id=sid)
        recibidos = [f"Carrera (desde malla adjunta): {carrera}"]
        if semestre: recibidos.append(f"Semestre: {semestre}")
        if jornada: recibidos.append(f"Jornada: {jornada}")

        msg = "Datos registrados en memoria hasta el momento:\n"
        for item in recibidos:
            msg += f"- {item}\n"
        msg += f"\nPara proceder con la generacion del boletin, por favor indicame: {', '.join(faltantes)}."
        return _finalizar_solicitud(t_total_ini, metricas, nuevo_estado, msg)

    if df_horarios.empty:
        metricas["tuvo_error"] = True
        registrar(
            "ERROR",
            "El Excel de horarios no esta cargado; no se puede generar la propuesta",
            "Sistema",
            duracion_ms=0.0,
            tokens=0,
            solicitud_id=sid,
            etapa="Motor"
        )
        return _finalizar_solicitud(
            t_total_ini,
            metricas,
            nuevo_estado,
            "No se pudo cargar el Excel de horarios, por lo que no es posible generar la propuesta."
        )

    t_motor_ini = time.perf_counter()
    try:
        propuesta = generar_propuesta(carrera, semestre, jornada)
        dur_motor_ms = round((time.perf_counter() - t_motor_ini) * 1000.0, 2)
        metricas["duracion_motor_ms"] = dur_motor_ms
    except Exception as e:
        dur_motor_ms = round((time.perf_counter() - t_motor_ini) * 1000.0, 2)
        metricas["duracion_motor_ms"] = dur_motor_ms
        metricas["tuvo_error"] = True
        registrar(
            "ERROR",
            f"Fallo del motor de horarios: {type(e).__name__}: {e}",
            "Motor de horarios",
            duracion_ms=dur_motor_ms,
            tokens=0,
            solicitud_id=sid,
            etapa="Motor"
        )
        return _finalizar_solicitud(
            t_total_ini,
            metricas,
            nuevo_estado,
            "Ocurrio un error al armar la propuesta. Revisa la pagina de logs o intenta nuevamente."
        )

    registrar_resultado_propuesta(propuesta, duracion_ms=dur_motor_ms, solicitud_id=sid)
    resumen = agente_redactor(propuesta, solicitud_id=sid, metricas_out=metricas)

    partes = [resumen]
    partes.append(f"### Asignaturas de la malla en el semestre {semestre}\n\n" + propuesta["tabla_malla"])
    if propuesta["tabla_horario"]:
        partes.append("### Horario propuesto\n\n" + propuesta["tabla_horario"])
        partes.append("### Detalle de secciones\n\n" + propuesta["tabla_detalle"])
    else:
        partes.append("No hay ninguna asignatura de este semestre con secciones publicadas en el Excel para la jornada indicada.")
    if propuesta["avisos"]:
        partes.append("### Asignaturas de la malla sin horario en esta propuesta\n\n" + "\n".join(propuesta["avisos"]))
    if propuesta["observaciones"]:
        partes.append("### Observaciones sobre los datos\n\n" + "\n".join(propuesta["observaciones"]))

    return _finalizar_solicitud(t_total_ini, metricas, nuevo_estado, "\n\n".join(partes))

# 9. ESTILOS VISUALES (HTML Y CSS)

Incluye los estilos institucionales (negro `#1A1A1A` y amarillo `#FFB81C`), las tarjetas de estado de la malla detectada (pendiente, aprobada o rechazada) y el resaltado al posar el mouse sobre el archivo adjunto para borrarlo con `×`.

In [9]:
estilos_css = """
:root {
    --duoc-yellow: #FFB81C;
    --duoc-black: #1A1A1A;
    --duoc-white: #FFFFFF;
}
body, .gradio-container {
    background-color: #EAECEF !important;
    font-family: 'Arial', sans-serif !important;
}
.gradio-container h1, .gradio-container h2, .gradio-container h3, .gradio-container p {
    color: var(--duoc-black) !important;
}
.encabezado-duoc {
    background-color: var(--duoc-black);
    padding: 25px;
    border-bottom: 5px solid var(--duoc-yellow);
    display: flex; align-items: center; justify-content: center;
    border-radius: 12px 12px 0 0;
}
.encabezado-duoc h1 { color: var(--duoc-yellow) !important; margin: 0; font-size: 28px; }
.encabezado-duoc p { color: #e0e0e0 !important; margin: 5px 0 0 0; }

/* Zona de carga de malla y accion de borrar al posar el mouse */
.subida-malla {
    border: 2px dashed var(--duoc-yellow) !important;
    border-radius: 10px !important;
    background-color: #FFFFFF !important;
    transition: all 0.2s ease;
}
.subida-malla:hover {
    border-color: #E09E00 !important;
    box-shadow: 0px 3px 10px rgba(0, 0, 0, 0.12);
}
.subida-malla table.file-preview tr:hover,
.subida-malla .file-preview:hover {
    background-color: #FFF8E1 !important;
}
.subida-malla button[aria-label="Clear"],
.subida-malla .file-preview button {
    opacity: 0.35;
    transition: opacity 0.2s ease, background-color 0.2s ease, color 0.2s ease;
    border-radius: 50% !important;
}
.subida-malla:hover button[aria-label="Clear"],
.subida-malla .file-preview:hover button {
    opacity: 1 !important;
    background-color: #FF5252 !important;
    color: #FFFFFF !important;
    cursor: pointer !important;
}

/* Tarjeta de estado de malla detectada en el panel izquierdo */
.estado-malla-card {
    background-color: var(--duoc-black);
    border-radius: 10px;
    padding: 14px 16px;
    margin-top: 8px;
    margin-bottom: 12px;
    box-shadow: 0px 3px 8px rgba(0, 0, 0, 0.22);
}
.estado-malla-pendiente {
    border: 2px solid #666666;
}
.estado-malla-ok {
    border: 2px solid var(--duoc-yellow);
}
.estado-malla-error {
    border: 2px solid #FF5252;
}
.estado-malla-titulo {
    font-size: 15px;
    font-weight: bold;
    color: var(--duoc-yellow) !important;
}
.estado-malla-error .estado-malla-titulo {
    color: #FF5252 !important;
}
.estado-malla-subtitulo {
    font-size: 13px;
    color: #FFFFFF !important;
    margin-top: 3px;
}
.estado-malla-detalle {
    font-size: 12px;
    color: #D0D0D0 !important;
    margin-top: 4px;
}
.estado-malla-tip {
    font-size: 11px;
    color: #9E9E9E !important;
    margin-top: 6px;
    border-top: 1px solid #333333;
    padding-top: 5px;
}

.caja-negra {
    background-color: var(--duoc-black) !important;
    color: var(--duoc-white) !important;
    padding: 20px !important;
    border-radius: 10px !important;
    border: 2px solid var(--duoc-yellow) !important;
    box-shadow: 0px 4px 10px rgba(0,0,0,0.3);
}
.caja-negra h1, .caja-negra h2, .caja-negra h3, .caja-negra p, .caja-negra li, .caja-negra span {
    color: var(--duoc-white) !important;
}
.caja-negra table {
    background-color: #2D2D2D !important;
    color: var(--duoc-white) !important;
    border-color: #444 !important;
}
.caja-negra th {
    background-color: #000000 !important;
    color: var(--duoc-yellow) !important;
    text-align: center !important;
}
.caja-negra td {
    background-color: #2D2D2D !important;
    color: var(--duoc-white) !important;
    border-color: #444 !important;
    text-align: center !important;
}
.caja-negra td * { color: var(--duoc-white) !important; }
"""

html_cabecera = """
<div class="encabezado-duoc">
    <div style="text-align: center;">
        <h1>Portal Academico de Estudiantes</h1>
        <p>Buscador Inteligente de Horarios y Boletin Academico</p>
    </div>
</div>
"""

# 10. INTERFAZ GRÁFICA CON CARGA DE MALLA Y MEMORIA DE ESTADO

Interfaz principal en Gradio:
1. **Panel izquierdo (Paso 1):** permite adjuntar una imagen `.JPG` o `.PNG` de la malla, muestra una **barra de carga** mientras ejecuta el OCR, identifica automáticamente la carrera en la tarjeta de estado y permite borrar el archivo al posar el mouse sobre él (`×`).
2. **Panel izquierdo (Paso 2):** campo de texto para indicar semestre y jornada.
3. **Panel derecho:** boletín académico generado.

In [10]:
with gr.Blocks(css=estilos_css) as portal_simulado:
    gr.HTML(html_cabecera)

    estado_memoria = gr.State({
        "carrera": None,
        "semestre": None,
        "jornada": None,
        "malla_cargada": False
    })

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 1. Adjunta tu malla curricular (.JPG o .PNG)")
            archivo_malla = gr.File(
                label="Arrastra tu malla aqui o haz clic para subir (.jpg, .png)",
                file_types=[".jpg", ".jpeg", ".png"],
                file_count="single",
                type="filepath",
                elem_classes=["subida-malla"]
            )
            tarjeta_estado_malla = gr.HTML(html_tarjeta_malla_vacia())

            gr.Markdown("### 2. Ingresa tu solicitud")
            entrada_usuario = gr.Textbox(
                label="",
                placeholder="Ej: Paso a 4to semestre en jornada Diurna.",
                lines=3,
                container=False
            )
            boton_generar = gr.Button("Enviar / Actualizar", variant="primary")

        with gr.Column(scale=2):
            gr.Markdown("### Tu Boletin Academico")
            salida_bot = gr.Markdown(
                "Adjunta tu malla curricular en el panel izquierdo e indica tu semestre y jornada para generar tu planificacion.",
                elem_classes=["caja-negra"]
            )

    # Al subir un archivo: ejecuta OCR con barra de carga, valida que sea malla y detecta carrera
    archivo_malla.upload(
        fn=procesar_subida_malla,
        inputs=[archivo_malla, estado_memoria],
        outputs=[archivo_malla, tarjeta_estado_malla, estado_memoria]
    )

    # Al posar el mouse sobre el archivo analizado y presionar (×): borra la malla y reinicia carrera
    archivo_malla.clear(
        fn=limpiar_malla_subida,
        inputs=[estado_memoria],
        outputs=[tarjeta_estado_malla, estado_memoria]
    )

    boton_generar.click(
        fn=procesar_solicitud,
        inputs=[entrada_usuario, estado_memoria],
        outputs=[salida_bot, estado_memoria]
    )

/tmp/ipykernel_565/2509957233.py:1: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=estilos_css) as portal_simulado:


# 11. LANZAMIENTO DEL PORTAL DE ESTUDIANTES

In [11]:
print("Generando link del portal de estudiantes en funcionamiento....")
portal_simulado.launch(share=True)

Generando link del portal de estudiantes en funcionamiento....
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7ed53ed362256aade5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# 12. SET DE PRUEBAS AUTOMATIZADAS (PRECISIÓN Y CONSISTENCIA)

**¿Qué hace esta celda?**
Define un conjunto de **18 mensajes de prueba** con variaciones reales (ordinales en palabras, abreviaciones `"3er sem"`, siglas `"IA"`, errores de tipeo, mensajes vacíos, solicitudes incompletas y semestres fuera de rango) más validación del requisito de malla adjunta.
Calcula **precisión global (`aciertos / total`)**, **precisión por campo** (`Carrera`, `Semestre`, `Jornada`) y **consistencia** repitiendo cada caso `N = 3` veces.

In [12]:
SET_PRUEBAS = [
    {
        "id": "P01",
        "categoria": "Estándar completo",
        "mensaje": "Hola, estudio Desarrollo de Software y paso a tercer semestre en jornada diurna",
        "esperado": {"carrera": "Desarrollo", "semestre": 3, "jornada": "Diurno"}
    },
    {
        "id": "P02",
        "categoria": "Ordinal en texto ('tercero')",
        "mensaje": "Soy de Mecánica, voy en tercero vespertino",
        "esperado": {"carrera": "Mecánica", "semestre": 3, "jornada": "Vespertino"}
    },
    {
        "id": "P03",
        "categoria": "Abreviación ('3er sem')",
        "mensaje": "Finanzas 3er sem diurno",
        "esperado": {"carrera": "Finanzas", "semestre": 3, "jornada": "Diurno"}
    },
    {
        "id": "P04",
        "categoria": "Sigla ('IA')",
        "mensaje": "Quiero ver mi horario de IA, 5to semestre diurno",
        "esperado": {"carrera": "Inteligencia Artificial", "semestre": 5, "jornada": "Diurno"}
    },
    {
        "id": "P05",
        "categoria": "Sigla ('IA') + vespertino",
        "mensaje": "Voy en cuarto semestre de IA en jornada vespertina",
        "esperado": {"carrera": "Inteligencia Artificial", "semestre": 4, "jornada": "Vespertino"}
    },
    {
        "id": "P06",
        "categoria": "Error de tipeo ('desarollo', 'diunro')",
        "mensaje": "hola soi de desarollo semestre 2 jornada diunro",
        "esperado": {"carrera": "Desarrollo", "semestre": 2, "jornada": "Diurno"}
    },
    {
        "id": "P07",
        "categoria": "Error de tipeo ('mecanica automotris', 'vespertno')",
        "mensaje": "necesito horario mecanica automotris 4to semestre vespertno",
        "esperado": {"carrera": "Mecánica", "semestre": 4, "jornada": "Vespertino"}
    },
    {
        "id": "P08",
        "categoria": "Error de tipeo ('finansas', 'segndo semstre')",
        "mensaje": "finansas segndo semstre diurno porfa",
        "esperado": {"carrera": "Finanzas", "semestre": 2, "jornada": "Diurno"}
    },
    {
        "id": "P09",
        "categoria": "Mensaje vacío",
        "mensaje": "",
        "esperado": {"carrera": None, "semestre": None, "jornada": None}
    },
    {
        "id": "P10",
        "categoria": "Mensaje solo espacios",
        "mensaje": "   \n\t  ",
        "esperado": {"carrera": None, "semestre": None, "jornada": None}
    },
    {
        "id": "P11",
        "categoria": "Ciencia de Datos + ordinal ('primer')",
        "mensaje": "Estudio Ciencia de Datos, primer semestre diurno",
        "esperado": {"carrera": "Ciencia de Datos", "semestre": 1, "jornada": "Diurno"}
    },
    {
        "id": "P12",
        "categoria": "Ingeniería de Datos + '6to sem'",
        "mensaje": "Ingeniería de Datos 6to sem vespertino",
        "esperado": {"carrera": "Ingeniería de Datos", "semestre": 6, "jornada": "Vespertino"}
    },
    {
        "id": "P13",
        "categoria": "Typo ('automatisacion') + octavo",
        "mensaje": "horario de automatisacion octavo semestre diurno",
        "esperado": {"carrera": "Automatización", "semestre": 8, "jornada": "Diurno"}
    },
    {
        "id": "P14",
        "categoria": "Solicitud incompleta (sin jornada)",
        "mensaje": "Hola, soy de Ciencia de Datos y paso a tercer semestre",
        "esperado": {"carrera": "Ciencia de Datos", "semestre": 3, "jornada": None}
    },
    {
        "id": "P15",
        "categoria": "Solicitud incompleta (solo jornada)",
        "mensaje": "¿y cómo queda en vespertino?",
        "esperado": {"carrera": None, "semestre": None, "jornada": "Vespertino"}
    },
    {
        "id": "P16",
        "categoria": "Semestre fuera de malla (semestre 10)",
        "mensaje": "Soy de Desarrollo, semestre 10 diurno",
        "esperado": {"carrera": "Desarrollo", "semestre": None, "jornada": "Diurno"}
    },
    {
        "id": "P17",
        "categoria": "Jerga y typo ('softwre', '7mo sem')",
        "mensaje": "q onda, softwre 7mo sem diurno",
        "esperado": {"carrera": "Desarrollo", "semestre": 7, "jornada": "Diurno"}
    },
    {
        "id": "P18",
        "categoria": "Saludo sin datos académicos",
        "mensaje": "Hola buenos días, quiero consultar sobre los horarios",
        "esperado": {"carrera": None, "semestre": None, "jornada": None}
    }
]

RESULTADOS_PRUEBAS = {
    "ejecutado": False,
    "precision": {"Carrera": 0.0, "Semestre": 0.0, "Jornada": 0.0, "Global": 0.0},
    "aciertos_global": 0,
    "total_casos": len(SET_PRUEBAS),
    "consistencia_promedio": 0.0,
    "casos_100_consistentes": 0,
    "n_repeticiones": 3,
    "detalle": []
}


def ejecutar_suite_pruebas(n_repeticiones=3, pausa_seg=0.25):
    """Ejecuta los 18 casos de prueba para evaluar precision y consistencia del Agente 1
    y la respuesta del flujo ante solicitudes con o sin malla adjunta."""
    assert 3 <= n_repeticiones <= 5, "n_repeticiones debe estar entre 3 y 5"
    detalle = []
    aciertos_global = 0
    aciertos_campo = {"carrera": 0, "semestre": 0, "jornada": 0}
    consistencias = []

    print(f"Ejecutando {len(SET_PRUEBAS)} casos de prueba (precision + consistencia x{n_repeticiones})...")
    for idx, caso in enumerate(SET_PRUEBAS, 1):
        msg = caso["mensaje"]
        esp = caso["esperado"]

        # Repeticion 1: evaluacion con procesar_solicitud (sin malla precargada, valida requisito de malla)
        _, estado_rep1 = procesar_solicitud(
            msg,
            {"carrera": None, "semestre": None, "jornada": None, "malla_cargada": False}
        )
        # Para validar semestre fuera de rango (1..8)
        if estado_rep1.get("semestre") and not (1 <= int(estado_rep1["semestre"]) <= 8):
            estado_rep1["semestre"] = None

        tupla_rep1 = (estado_rep1.get("carrera"), estado_rep1.get("semestre"), estado_rep1.get("jornada"))

        ok_c = (estado_rep1.get("carrera") == esp["carrera"])
        ok_s = (estado_rep1.get("semestre") == esp["semestre"])
        ok_j = (estado_rep1.get("jornada") == esp["jornada"])
        ok_all = ok_c and ok_s and ok_j

        if ok_c: aciertos_campo["carrera"] += 1
        if ok_s: aciertos_campo["semestre"] += 1
        if ok_j: aciertos_campo["jornada"] += 1
        if ok_all: aciertos_global += 1

        # Repeticiones adicionales (2..n_repeticiones) para consistencia
        predicciones = [tupla_rep1]
        for rep in range(1, n_repeticiones):
            if pausa_seg > 0:
                time.sleep(pausa_seg)
            if not str(msg or "").strip():
                predicciones.append((None, None, None))
            else:
                st = agente_analista(msg, {"carrera": None, "semestre": None, "jornada": None, "malla_cargada": False})
                if st.get("semestre") and not (1 <= int(st["semestre"]) <= 8):
                    st["semestre"] = None
                predicciones.append((st.get("carrera"), st.get("semestre"), st.get("jornada")))

        conteo_pred = Counter(predicciones)
        moda, freq_moda = conteo_pred.most_common(1)[0]
        consistencia_caso = freq_moda / n_repeticiones
        consistencias.append(consistencia_caso)

        detalle.append({
            "id": caso["id"],
            "categoria": caso["categoria"],
            "mensaje": msg if msg.strip() else repr(msg),
            "esperado": str(esp),
            "obtenido": str({"carrera": tupla_rep1[0], "semestre": tupla_rep1[1], "jornada": tupla_rep1[2]}),
            "acierto_carrera": ok_c,
            "acierto_semestre": ok_s,
            "acierto_jornada": ok_j,
            "acierto_global": ok_all,
            "coincidencias": f"{freq_moda}/{n_repeticiones}",
            "consistencia": round(consistencia_caso, 4)
        })
        if pausa_seg > 0:
            time.sleep(pausa_seg)

    total = len(SET_PRUEBAS)
    RESULTADOS_PRUEBAS.update({
        "ejecutado": True,
        "precision": {
            "Carrera": round(aciertos_campo["carrera"] / total, 4),
            "Semestre": round(aciertos_campo["semestre"] / total, 4),
            "Jornada": round(aciertos_campo["jornada"] / total, 4),
            "Global": round(aciertos_global / total, 4),
        },
        "aciertos_global": aciertos_global,
        "total_casos": total,
        "consistencia_promedio": round(float(np.mean(consistencias)), 4),
        "casos_100_consistentes": sum(1 for c in consistencias if c == 1.0),
        "n_repeticiones": n_repeticiones,
        "detalle": detalle
    })
    print(f"[Exito] Pruebas completadas: Precision Global = {aciertos_global}/{total} ({RESULTADOS_PRUEBAS['precision']['Global']*100:.1f}%), "
          f"Consistencia promedio = {RESULTADOS_PRUEBAS['consistencia_promedio']*100:.1f}%")
    return pd.DataFrame(detalle)

print(f"[Exito] Set de pruebas cargado con {len(SET_PRUEBAS)} casos.")

[Exito] Set de pruebas cargado con 18 casos.


# 13. PÁGINA DE LOGS Y DASHBOARD DE OBSERVABILIDAD (GRADIO + PLOTLY)

Muestra en tiempo real las tarjetas KPI, los 4 gráficos interactivos con Plotly (latencia por solicitud, errores por origen, precisión por campo y tokens acumulados) y la bitácora filtrable.

In [13]:
from html import escape

COLORES_NIVEL = {"INFO": "#4FC3F7", "WARNING": "#FFB81C", "ERROR": "#FF5252"}
ORIGENES = ["Todos", "Sistema", "Agente 1", "Agente 2", "Motor de horarios", "Lectura de mallas"]


def copiar_logs(nivel="Todos", origen="Todos"):
    with CANDADO_LOGS:
        entradas = list(REGISTRO_LOGS)
    if nivel != "Todos":
        entradas = [e for e in entradas if e["nivel"] == nivel]
    if origen != "Todos":
        entradas = [e for e in entradas if e["origen"] == origen]
    return entradas


def _estilo_figura_duoc(fig, titulo, xaxis_title="", yaxis_title=""):
    fig.update_layout(
        title=dict(text=titulo, font=dict(color="#FFB81C", size=15)),
        paper_bgcolor="#1A1A1A",
        plot_bgcolor="#111111",
        font=dict(color="#FFFFFF", size=12),
        xaxis=dict(title=xaxis_title, gridcolor="#2D2D2D", zerolinecolor="#444444"),
        yaxis=dict(title=yaxis_title, gridcolor="#2D2D2D", zerolinecolor="#444444"),
        legend=dict(bgcolor="rgba(0,0,0,0.4)", font=dict(color="#FFFFFF")),
        margin=dict(l=45, r=20, t=45, b=40),
        height=320
    )
    return fig


def fig_latencia_solicitud():
    with CANDADO_LOGS:
        reqs = list(REGISTRO_SOLICITUDES)
    fig = go.Figure()
    if not reqs:
        fig.add_annotation(text="Sin solicitudes registradas aun", showarrow=False, font=dict(color="#BDBDBD", size=14))
        return _estilo_figura_duoc(fig, "1. Latencia por Solicitud (ms)", "Solicitud #", "Duracion (ms)")

    xs = [f"Req #{r['solicitud_id']}" for r in reqs]
    fig.add_trace(go.Bar(name="Agente 1", x=xs, y=[r["duracion_agente1_ms"] for r in reqs], marker_color="#4FC3F7"))
    fig.add_trace(go.Bar(name="Motor", x=xs, y=[r["duracion_motor_ms"] for r in reqs], marker_color="#81C784"))
    fig.add_trace(go.Bar(name="Agente 2", x=xs, y=[r["duracion_agente2_ms"] for r in reqs], marker_color="#FFB81C"))
    fig.add_trace(go.Scatter(
        name="Total (ms)", x=xs, y=[r["duracion_total_ms"] for r in reqs],
        mode="lines+markers", line=dict(color="#FF5252", width=2)
    ))
    fig.update_layout(barmode="stack")
    return _estilo_figura_duoc(fig, "1. Latencia por Solicitud (ms)", "Solicitud", "Duracion (ms)")


def fig_errores_origen():
    todas = copiar_logs()
    origenes_lista = ["Sistema", "Agente 1", "Agente 2", "Motor de horarios", "Lectura de mallas"]
    errores = [sum(1 for e in todas if e["origen"] == o and e["nivel"] == "ERROR") for o in origenes_lista]
    warnings = [sum(1 for e in todas if e["origen"] == o and e["nivel"] == "WARNING") for o in origenes_lista]

    fig = go.Figure()
    fig.add_trace(go.Bar(name="ERROR", x=origenes_lista, y=errores, marker_color="#FF5252"))
    fig.add_trace(go.Bar(name="WARNING", x=origenes_lista, y=warnings, marker_color="#FFB81C"))
    fig.update_layout(barmode="group")
    return _estilo_figura_duoc(fig, "2. Errores y Advertencias por Origen", "Origen", "Cantidad de eventos")


def fig_precision_campo():
    fig = go.Figure()
    if not RESULTADOS_PRUEBAS.get("ejecutado"):
        fig.add_annotation(
            text="Ejecuta la suite de pruebas (Celda 12/14) para ver la precision por campo",
            showarrow=False, font=dict(color="#BDBDBD", size=13)
        )
        return _estilo_figura_duoc(fig, "3. Precision por Campo (%)", "Campo", "Precision (%)")

    prec = RESULTADOS_PRUEBAS["precision"]
    campos = ["Carrera", "Semestre", "Jornada", "Global"]
    valores = [round(prec[c] * 100, 1) for c in campos]
    colores = ["#4FC3F7", "#81C784", "#BA68C8", "#FFB81C"]
    fig.add_trace(go.Bar(
        x=campos,
        y=valores,
        text=[f"{v}%" for v in valores],
        textposition="auto",
        marker_color=colores
    ))
    fig.update_yaxes(range=[0, 105])
    return _estilo_figura_duoc(fig, "3. Precision por Campo (%)", "Campo evaluado", "Precision (%)")


def fig_tokens_acumulados():
    todas = copiar_logs()
    eventos_tok = [e for e in todas if (e.get("tokens") or 0) > 0 and e.get("etapa") in ("Agente 1", "Agente 2")]
    fig = go.Figure()
    if not eventos_tok:
        fig.add_annotation(text="Sin llamadas LLM con tokens registrados aun", showarrow=False, font=dict(color="#BDBDBD", size=14))
        return _estilo_figura_duoc(fig, "4. Tokens Acumulados", "Llamada LLM #", "Tokens")

    xs = [f"#{i+1} ({e['etapa']})" for i, e in enumerate(eventos_tok)]
    toks_step = [e["tokens"] for e in eventos_tok]
    toks_acum = list(np.cumsum(toks_step))

    fig.add_trace(go.Scatter(
        name="Tokens acumulados", x=xs, y=toks_acum,
        mode="lines+markers", fill="tozeroy", line=dict(color="#FFB81C", width=3)
    ))
    fig.add_trace(go.Bar(
        name="Tokens por llamada", x=xs, y=toks_step,
        marker_color="#4FC3F7", opacity=0.6
    ))
    return _estilo_figura_duoc(fig, "4. Tokens Acumulados (usage.total_tokens)", "Llamada LLM", "Tokens")


def construir_resumen_html():
    todas = copiar_logs()
    m = calcular_metricas_logs()
    conteo = {n: sum(1 for e in todas if e["nivel"] == n) for n in COLORES_NIVEL}
    freq_err_pct = f"{m['frecuencia_errores']*100:.1f}% ({m['errores_totales']}/{m['solicitudes_totales']})"
    tasa_resp_pct = f"{m['tasa_resumen_respaldo']*100:.1f}% ({m['usos_resumen_respaldo']}/{m['llamadas_agente2']})"
    lat_media = f"{m['latencia_etapas']['Total']['promedio_ms']:.1f} ms"

    tarjetas = [
        ("Total de eventos", len(todas), "#FFFFFF"),
        ("INFO", conteo["INFO"], COLORES_NIVEL["INFO"]),
        ("WARNING", conteo["WARNING"], COLORES_NIVEL["WARNING"]),
        ("ERROR", conteo["ERROR"], COLORES_NIVEL["ERROR"]),
        ("Frecuencia errores", freq_err_pct, "#FF8A65"),
        ("Uso resumen respaldo", tasa_resp_pct, "#FFB81C"),
        ("Latencia media Total", lat_media, "#81C784"),
        ("Tokens acumulados", m["tokens_totales"], "#4FC3F7"),
    ]
    cuerpo = "".join(
        f"<div class='tarjeta-log'><div class='tarjeta-valor' style='color:{color};font-size:22px'>{valor}</div>"
        f"<div class='tarjeta-titulo'>{titulo}</div></div>"
        for titulo, valor, color in tarjetas
    )
    return f"<div class='fila-tarjetas'>{cuerpo}</div>"


def construir_lista_html(nivel="Todos", origen="Todos"):
    entradas = copiar_logs(nivel, origen)
    if not entradas:
        return ("<div class='caja-logs'><div class='log-vacio'>Aun no hay eventos que coincidan. "
                "Usa el portal de estudiantes o ejecuta las pruebas y vuelve a esta pagina.</div></div>")
    filas = []
    for e in reversed(entradas):
        color = COLORES_NIVEL.get(e["nivel"], "#FFFFFF")
        dur_badge = f"<span class='log-metrica'>{e['duracion_ms']} ms</span>" if e.get("duracion_ms") is not None else ""
        tok_badge = f"<span class='log-metrica'>{e['tokens']} tok</span>" if (e.get("tokens") or 0) > 0 else ""
        filas.append(
            "<div class='log-linea'>"
            f"<span class='log-fecha'>{e['fecha']}</span>"
            f"<span class='log-nivel' style='color:{color};border-color:{color}'>{e['nivel']}</span>"
            f"<span class='log-origen'>{escape(e['origen'])}</span>"
            f"<span class='log-mensaje'>{escape(e['mensaje'])}</span>"
            f"{dur_badge}{tok_badge}"
            "</div>"
        )
    return "<div class='caja-logs'>" + "".join(filas) + "</div>"


def refrescar_logs(nivel, origen):
    return (
        construir_resumen_html(),
        fig_latencia_solicitud(),
        fig_errores_origen(),
        fig_precision_campo(),
        fig_tokens_acumulados(),
        construir_lista_html(nivel, origen)
    )


def accion_exportar_logs():
    ruta_json, _ = exportar_logs()
    return ruta_json


def mostrar_logs_consola():
    entradas = copiar_logs()
    if not entradas:
        print("[Aviso] Aun no hay eventos registrados.")
        return
    for e in entradas:
        extra = ""
        if e.get("duracion_ms") is not None:
            extra += f" | duracion_ms={e['duracion_ms']}"
        if (e.get("tokens") or 0) > 0:
            extra += f" | tokens={e['tokens']}"
        print(f"{e['fecha']} | {e['nivel']:<7} | {e['origen']} | {e['mensaje']}{extra}")


estilos_logs = estilos_css + """
.fila-tarjetas { display: flex; gap: 10px; flex-wrap: wrap; margin-bottom: 10px; }
.tarjeta-log { flex: 1; min-width: 140px; background: #1A1A1A; border: 2px solid #FFB81C; border-radius: 10px; padding: 12px; text-align: center; box-shadow: 0px 4px 10px rgba(0,0,0,0.25); }
.tarjeta-valor { font-size: 24px; font-weight: bold; }
.tarjeta-titulo { color: #e0e0e0; font-size: 12px; margin-top: 4px; }
.caja-logs { background: #111111; border: 2px solid #FFB81C; border-radius: 10px; padding: 14px; max-height: 460px; overflow-y: auto; font-family: 'Courier New', monospace; font-size: 13px; }
.log-linea { padding: 6px 4px; border-bottom: 1px solid #2D2D2D; display: flex; gap: 10px; flex-wrap: wrap; align-items: baseline; }
.log-fecha { color: #9E9E9E; }
.log-nivel { border: 1px solid; border-radius: 4px; padding: 0 6px; font-weight: bold; min-width: 70px; text-align: center; }
.log-origen { color: #FFB81C; min-width: 130px; }
.log-mensaje { color: #FFFFFF; flex: 1; }
.log-metrica { background: #2D2D2D; color: #81C784; border: 1px solid #444; border-radius: 4px; padding: 1px 6px; font-size: 11px; }
.log-vacio { color: #BDBDBD; padding: 20px; text-align: center; }
"""

html_cabecera_logs = """
<div class="encabezado-duoc">
    <div style="text-align: center;">
        <h1>Dashboard de Observabilidad y Registro de Logs</h1>
        <p>Latencia por etapa, tokens consumidos, precision, errores y eventos en tiempo real</p>
    </div>
</div>
"""

with gr.Blocks(css=estilos_logs) as portal_logs:
    gr.HTML(html_cabecera_logs)

    with gr.Row():
        filtro_nivel = gr.Dropdown(["Todos", "INFO", "WARNING", "ERROR"], value="Todos", label="Nivel")
        filtro_origen = gr.Dropdown(ORIGENES, value="Todos", label="Origen")
        boton_actualizar = gr.Button("Actualizar Dashboard", variant="primary")
        boton_exportar = gr.Button("Exportar Logs (.json)")

    archivo_descarga = gr.File(label="Archivo exportado", visible=True)
    salida_resumen = gr.HTML()

    with gr.Row():
        plot_latencia = gr.Plot(label="Latencia por solicitud")
        plot_errores = gr.Plot(label="Errores por origen")

    with gr.Row():
        plot_precision = gr.Plot(label="Precision por campo")
        plot_tokens = gr.Plot(label="Tokens acumulados")

    salida_lista = gr.HTML()

    entradas_filtros = [filtro_nivel, filtro_origen]
    salidas_logs = [salida_resumen, plot_latencia, plot_errores, plot_precision, plot_tokens, salida_lista]

    boton_actualizar.click(fn=refrescar_logs, inputs=entradas_filtros, outputs=salidas_logs)
    boton_exportar.click(fn=accion_exportar_logs, inputs=[], outputs=[archivo_descarga])
    filtro_nivel.change(fn=refrescar_logs, inputs=entradas_filtros, outputs=salidas_logs)
    filtro_origen.change(fn=refrescar_logs, inputs=entradas_filtros, outputs=salidas_logs)
    portal_logs.load(fn=refrescar_logs, inputs=entradas_filtros, outputs=salidas_logs)

    if hasattr(gr, "Timer"):
        temporizador = gr.Timer(5)
        temporizador.tick(fn=refrescar_logs, inputs=entradas_filtros, outputs=salidas_logs)

registrar("INFO", "Pagina de logs y dashboard iniciados", "Sistema")
print("Generando link de la pagina de logs y dashboard....")
try:
    portal_logs.launch(share=True)
except Exception as e:
    print(f"[Aviso] No se pudo abrir la pagina de logs: {e}")
    print("Mostrando el registro en consola:\n")
    mostrar_logs_consola()

/tmp/ipykernel_565/2677973055.py:214: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=estilos_logs) as portal_logs:


Generando link de la pagina de logs y dashboard....
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://670d75d2dabb1e9818.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# 14. EJECUCIÓN DE PRUEBAS, EXPORTACIÓN DE LOGS Y ANÁLISIS REAL

Ejecuta la suite de 18 pruebas (precisión y consistencia x3 repeticiones), exporta los logs a `.json`, `.csv` y `.log`, e imprime el análisis automático.

In [14]:
def analizar_logs_reales():
    metricas = calcular_metricas_logs()
    with CANDADO_LOGS:
        logs = list(REGISTRO_LOGS)

    df_logs = pd.DataFrame(logs)
    warnings_df = df_logs[df_logs["nivel"] == "WARNING"] if not df_logs.empty else pd.DataFrame()

    conteo_siglas_warning = Counter()
    detalle_siglas_warning = {}
    if not warnings_df.empty:
        for _, fila in warnings_df.iterrows():
            msg = str(fila["mensaje"])
            siglas_encontradas = re.findall(r"[A-Z]{3}\d{4}", msg)
            for sg in siglas_encontradas:
                conteo_siglas_warning[sg] += 1
                nombre_asig = catalogo_siglas.get(sg, "Nombre no catalogado en Excel")
                detalle_siglas_warning[sg] = nombre_asig

    lat = metricas["latencia_etapas"]
    etapas_comparables = {k: lat[k]["promedio_ms"] for k in ("Agente 1", "Motor", "Agente 2")}
    cuello_botella = max(etapas_comparables, key=etapas_comparables.get) if any(etapas_comparables.values()) else "Sin datos"

    print("=" * 72)
    print("REPORTE DE ANALISIS DE LOGS REALES Y EVALUACION")
    print("=" * 72)
    print(f"1. PRECISION Y CONSISTENCIA (Set de {RESULTADOS_PRUEBAS['total_casos']} pruebas, {RESULTADOS_PRUEBAS['n_repeticiones']} repeticiones):")
    print(f"   - Precision Global (aciertos/total): {RESULTADOS_PRUEBAS['aciertos_global']}/{RESULTADOS_PRUEBAS['total_casos']} "
          f"({RESULTADOS_PRUEBAS['precision']['Global']*100:.2f}%)")
    print(f"   - Precision por campo -> Carrera: {RESULTADOS_PRUEBAS['precision']['Carrera']*100:.2f}% | "
          f"Semestre: {RESULTADOS_PRUEBAS['precision']['Semestre']*100:.2f}% | "
          f"Jornada: {RESULTADOS_PRUEBAS['precision']['Jornada']*100:.2f}%")
    print(f"   - Consistencia promedio: {RESULTADOS_PRUEBAS['consistencia_promedio']*100:.2f}% "
          f"({RESULTADOS_PRUEBAS['casos_100_consistentes']}/{RESULTADOS_PRUEBAS['total_casos']} casos 100% consistentes)")

    print("\n2. ERRORES Y FILTRO DE SEGURIDAD:")
    print(f"   - Solicitudes totales: {metricas['solicitudes_totales']}")
    print(f"   - Errores totales: {metricas['errores_totales']}")
    print(f"   - Frecuencia de errores (errores / solicitudes): {metricas['frecuencia_errores']*100:.2f}%")
    print(f"   - Tasa de uso resumen de respaldo: {metricas['tasa_resumen_respaldo']*100:.2f}% "
          f"({metricas['usos_resumen_respaldo']}/{metricas['llamadas_agente2']} llamadas Agente 2)")

    print("\n3. LATENCIA POR ETAPA Y CUELLO DE BOTELLA:")
    for et in ("Agente 1", "Motor", "Agente 2", "Total"):
        info_et = lat[et]
        print(f"   - {et:<10}: promedio={info_et['promedio_ms']:.2f} ms | mediana={info_et['mediana_ms']:.2f} ms | "
              f"p95={info_et['p95_ms']:.2f} ms (n={info_et['n']})")
    print(f"   >>> CUELLO DE BOTELLA IDENTIFICADO: {cuello_botella} ({etapas_comparables.get(cuello_botella, 0):.2f} ms promedio)")

    print("\n4. CONSUMO DE RECURSOS (TOKENS):")
    print(f"   - Tokens Agente 1: {metricas['tokens_agente1']}")
    print(f"   - Tokens Agente 2: {metricas['tokens_agente2']}")
    print(f"   - Tokens Totales : {metricas['tokens_totales']}")

    print("\n5. ASIGNATURAS CON MAS EVENTOS WARNING:")
    if conteo_siglas_warning:
        for sg, cnt in conteo_siglas_warning.most_common(10):
            print(f"   - {sg} ({detalle_siglas_warning.get(sg, '')}): {cnt} apariciones en WARNING")
    else:
        print("   - No se registraron siglas en eventos WARNING durante esta ejecucion.")
    print("=" * 72)

    return {
        "metricas": metricas,
        "cuello_botella": cuello_botella,
        "top_warnings_siglas": conteo_siglas_warning.most_common(10)
    }


df_resultados_pruebas = ejecutar_suite_pruebas(n_repeticiones=3, pausa_seg=0.25)
ruta_json_exp, ruta_csv_exp = exportar_logs()
resumen_analisis = analizar_logs_reales()

display(df_resultados_pruebas[["id", "categoria", "mensaje", "acierto_global", "coincidencias", "consistencia"]])

try:
    from google.colab import files
    files.download(ruta_json_exp)
    files.download(ARCHIVO_LOGS)
except Exception:
    print(f"[Info] Logs exportados localmente en: {ruta_json_exp}, {ruta_csv_exp} y {ARCHIVO_LOGS}")

Ejecutando 18 casos de prueba (precision + consistencia x3)...
[Exito] Pruebas completadas: Precision Global = 15/18 (83.3%), Consistencia promedio = 100.0%
REPORTE DE ANALISIS DE LOGS REALES Y EVALUACION
1. PRECISION Y CONSISTENCIA (Set de 18 pruebas, 3 repeticiones):
   - Precision Global (aciertos/total): 15/18 (83.33%)
   - Precision por campo -> Carrera: 83.33% | Semestre: 100.00% | Jornada: 94.44%
   - Consistencia promedio: 100.00% (18/18 casos 100% consistentes)

2. ERRORES Y FILTRO DE SEGURIDAD:
   - Solicitudes totales: 20
   - Errores totales: 0
   - Frecuencia de errores (errores / solicitudes): 0.00%
   - Tasa de uso resumen de respaldo: 0.00% (0/2 llamadas Agente 2)

3. LATENCIA POR ETAPA Y CUELLO DE BOTELLA:
   - Agente 1  : promedio=4010.71 ms | mediana=4745.05 ms | p95=6941.38 ms (n=50)
   - Motor     : promedio=201.25 ms | mediana=201.25 ms | p95=221.30 ms (n=2)
   - Agente 2  : promedio=1301.19 ms | mediana=1301.19 ms | p95=1416.50 ms (n=2)
   - Total     : promedio=

,id,categoria,mensaje,acierto_global,coincidencias,consistencia
0,P01,Estándar completo,"Hola, estudio Desarrollo de Software y paso a ...",True,3/3,1.0
1,P02,Ordinal en texto ('tercero'),"Soy de Mecánica, voy en tercero vespertino",True,3/3,1.0
2,P03,Abreviación ('3er sem'),Finanzas 3er sem diurno,True,3/3,1.0
3,P04,Sigla ('IA'),"Quiero ver mi horario de IA, 5to semestre diurno",True,3/3,1.0
4,P05,Sigla ('IA') + vespertino,Voy en cuarto semestre de IA en jornada vesper...,True,3/3,1.0
5,P06,"Error de tipeo ('desarollo', 'diunro')",hola soi de desarollo semestre 2 jornada diunro,False,3/3,1.0
6,P07,"Error de tipeo ('mecanica automotris', 'vesper...",necesito horario mecanica automotris 4to semes...,True,3/3,1.0
7,P08,"Error de tipeo ('finansas', 'segndo semstre')",finansas segndo semstre diurno porfa,False,3/3,1.0
8,P09,Mensaje vacío,'',True,3/3,1.0
9,P10,Mensaje solo espacios,' \n\t ',True,3/3,1.0


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>